# SECTION: SPATIAL DISTANCE ANALYSIS
# Quantifies the physical proximity of pericytes and
 islet-associated fibroblasts to key anatomical landmarks:

   - dist_edge_endo_um  : distance to nearest endothelial cell
   - dist_edge_islet_um : distance to islet boundary

 Approach:
   1. Paired Wilcoxon test (donor-level medians) comparing
      pericyte vs islet-associated fibroblast distance
      to the endothelial edge within islet compartment
   2. KDE and step-histogram plots of cell density
      relative to the islet boundary (±50 µm window)
   3. Boundary enrichment: % cells within ±5 µm per donor
      per cell type → exported as CSV for GraphPad Prism
   4. Distance-bin fractions (0–5, 5–10, 10–20, 20–40 µm)
      for pericytes relative to endothelium
   5. Pericyte–EC proximity: fraction of pericytes within
      2 µm of an endothelial cell (coverage curve)

 Key objects: vasc_healthy, islet_adata_vasc

In [ ]:
import concord as ccd
import scanpy as sc
import torch
import os
import pandas as pd
import numpy as np
import anndata as ad

In [ ]:
vasc_healthy = sc.read_h5ad("/Users/kmeneses/Desktop/updated_data/healthyvascularcells_allsamples_no9317162_labeledCV.h5ad")

In [ ]:
vasc_healthy

In [ ]:
vasc_healthy.obs['Location'].value_counts()

In [ ]:
color_by = ['cell_type_final', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    vasc_healthy, basis='Concord_UMAPmRU', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data'
)

In [ ]:
df = vasc_healthy.obs.loc[
    (vasc_healthy.obs["Location"] == "Islet") &
    (vasc_healthy.obs["cell_type_final"].isin([
        "Pericytes",
        "Islet-associated Fibroblasts"
    ])),
    ["Sample", "cell_type_final", "dist_edge_endo_um"]
].copy()

In [ ]:
from scipy.stats import wilcoxon

donor_summary = (
    df.groupby(["Sample", "cell_type_final"])
      ["dist_edge_endo_um"]
      .median()
      .reset_index()
)

wide = donor_summary.pivot(
    index="Sample",
    columns="cell_type_final",
    values="dist_edge_endo_um"
)

print(wide)

stat, p = wilcoxon(
    wide["Pericytes"],
    wide["Islet-associated Fibroblasts"]
)

print(f"Paired Wilcoxon p = {p:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(2.2,3))

for donor in wide.index:
    plt.plot(
        [0,1],
        [
            wide.loc[donor, "Pericytes"],
            wide.loc[donor, "Islet-associated Fibroblasts"]
        ],
        color="lightgray",
        zorder=1
    )

plt.scatter(
    [0]*len(wide),
    wide["Pericytes"],
    s=40,
    zorder=2
)

plt.scatter(
    [1]*len(wide),
    wide["Islet-associated Fibroblasts"],
    s=40,
    zorder=2
)

plt.xticks(
    [0,1],
    ["Pericytes", "Islet-associated\nFibroblasts"]
)

plt.ylabel("Distance to endothelial edge (µm)")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# STATS
# =========================
stat, p = wilcoxon(
    wide["Pericytes"],
    wide["Islet-associated Fibroblasts"]
)

print(f"Paired Wilcoxon p = {p:.4f}")

# =========================
# SUMMARY
# =========================
means = [
    wide["Pericytes"].mean(),
    wide["Islet-associated Fibroblasts"].mean()
]

sems = [
    wide["Pericytes"].sem(),
    wide["Islet-associated Fibroblasts"].sem()
]

# =========================
# PLOT
# =========================
fig, ax = plt.subplots(figsize=(2.2,3))

ax.bar(
    [0,1],
    means,
    yerr=sems,
    capsize=3
)

# donor points
ax.scatter(
    np.zeros(len(wide)),
    wide["Pericytes"],
    zorder=3
)

ax.scatter(
    np.ones(len(wide)),
    wide["Islet-associated Fibroblasts"],
    zorder=3
)

# paired lines
for donor in wide.index:
    ax.plot(
        [0,1],
        [
            wide.loc[donor, "Pericytes"],
            wide.loc[donor, "Islet-associated Fibroblasts"]
        ],
        color="gray",
        alpha=0.6,
        zorder=2
    )

# significance bar
ymax = max(
    wide["Pericytes"].max(),
    wide["Islet-associated Fibroblasts"].max()
)

y = ymax * 1.10

ax.plot(
    [0,0,1,1],
    [y,y*1.03,y*1.03,y],
    lw=1,
    color="black"
)

ax.text(
    0.5,
    y*1.05,
    f"p = {p:.3f}",
    ha="center"
)

ax.set_xticks([0,1])
ax.set_xticklabels([
    "Pericytes",
    "Islet-associated\nFibroblasts"
])

ax.set_ylabel("Distance to endothelial edge (µm)")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import mannwhitneyu

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SUBSET
# =========================
df = vasc_healthy.obs.loc[
    (vasc_healthy.obs["Location"] == "Islet") &
    (vasc_healthy.obs["cell_type_final"].isin([
        "Pericytes",
        "Islet-associated Fibroblasts"
    ])),
    ["cell_type_final", "dist_edge_endo_um", "Sample"]
].copy()

# optional
df = df[df["dist_edge_endo_um"] >= 0]

# =========================
# STATS
# =========================
peri = df.loc[
    df["cell_type_final"] == "Pericytes",
    "dist_edge_endo_um"
]

fib = df.loc[
    df["cell_type_final"] == "Islet-associated Fibroblasts",
    "dist_edge_endo_um"
]

stat, p = mannwhitneyu(peri, fib)

print(f"MWU p = {p:.3e}")

# =========================
# PLOT
# =========================
plt.figure(figsize=(2.2,3))

sns.violinplot(
    data=df,
    x="cell_type_final",
    y="dist_edge_endo_um",
    inner=None,
    cut=0
)

sns.boxplot(
    data=df,
    x="cell_type_final",
    y="dist_edge_endo_um",
    width=0.2,
    showcaps=False,
    boxprops={"facecolor":"white"},
    showfliers=False
)

sns.stripplot(
    data=df.sample(
        min(5000, len(df)),
        random_state=1
    ),
    x="cell_type_final",
    y="dist_edge_endo_um",
    hue="Sample",
    size=2,
    alpha=0.5,
    dodge=False
)

plt.xlabel("")
plt.ylabel("Distance to endothelial edge (µm)")
plt.xticks(
    [0,1],
    ["Pericytes", "Islet-associated\nFibroblasts"]
)

plt.legend(
    bbox_to_anchor=(1.05,1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"

cell_types = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts"
]

# =========================
# DATA
# =========================
df = islet_adata_vasc.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].astype(str).str.strip()

# spatial window (important)

df = df[df[CELLTYPE_COL].isin(cell_types)]

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b"
    }

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,2))

sns.histplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    bins=40,
    stat="density",        # 🔥 IMPORTANT (not counts)
    common_norm=False,     # 🔥 each cell type normalized independently
    palette=palette,
    alpha=0.6,
     # cleaner than filled bars
    linewidth=1
)

# boundary line
plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.xlabel("Distance to Islet Boundary (µm)")
plt.ylabel("Relative cell density")
plt.title("Spatial distribution near islet boundary")

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
out = "/Users/kmeneses/Desktop/Figure_2_plots/SI/hist_boundary_distribution"

plt.savefig(out + ".png", dpi=600, bbox_inches="tight")
plt.savefig(out + ".svg", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"

cell_types = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts",
    "Fibroblasts"
]

# =========================
# DATA
# =========================
df = vasc_healthy.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].astype(str).str.strip()

# spatial window (important)
df = df[np.abs(df[DIST_COL]) <= 50]

df = df[df[CELLTYPE_COL].isin(cell_types)]

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b",
    "Fibroblasts": "#27d0ee"
}

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,2))

sns.histplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    bins=40,
    stat="density",        # 🔥 IMPORTANT (not counts)
    common_norm=False,     # 🔥 each cell type normalized independently
    palette=palette,
    alpha=0.6,
     # cleaner than filled bars
    linewidth=1
)

# boundary line
plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.xlabel("Distance to Islet Boundary (µm)")
plt.ylabel("Relative cell density")
plt.title("Spatial distribution near islet boundary")

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
out = "/Users/kmeneses/Desktop/Figure_2_plots/SI/hist_boundary_distribution"

plt.savefig(out + ".png", dpi=600, bbox_inches="tight")
plt.savefig(out + ".svg", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"

cell_types = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts",
    "Fibroblasts"
]

# =========================
# DATA
# =========================
df = vasc_healthy.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].astype(str).str.strip()

# spatial window
df = df[np.abs(df[DIST_COL]) <= 50]

df = df[df[CELLTYPE_COL].isin(cell_types)]

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b",
    "Fibroblasts": "#27d0ee"
}

# =========================
# PLOT (SMOOTH KDE)
# =========================
plt.figure(figsize=(3,2))

sns.kdeplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    palette=palette,
    common_norm=False,
    linewidth=1.5,
    fill=True,
    alpha=0.3,
    bw_adjust=0.8  # 🔥 controls smoothness (lower = sharper, higher = smoother)
)

# boundary line
plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.xlabel("Distance to Islet Boundary (µm)")
plt.ylabel("Relative cell density")
plt.title("Spatial distribution near islet boundary")

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
out = "/Users/kmeneses/Desktop/Figure_2_plots/SI/kde_boundary_distribution"

plt.savefig(out + ".png", dpi=600, bbox_inches="tight")
plt.savefig(out + ".svg", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"

cell_types = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts"
]

# =========================
# DATA
# =========================
df = islet_adata_vasc.obs[[DIST_COL, CELLTYPE_COL]].dropna().copy()

df[CELLTYPE_COL] = df[CELLTYPE_COL].astype(str).str.strip()

# spatial window
df = df[np.abs(df[DIST_COL]) <= 50]

df = df[df[CELLTYPE_COL].isin(cell_types)]

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b"
    }

# =========================
# PLOT (SMOOTH KDE)
# =========================
plt.figure(figsize=(3,2))

sns.kdeplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    palette=palette,
    common_norm=False,
    linewidth=1.5,
    fill=True,
    alpha=0.3,
    bw_adjust=0.8  # 🔥 controls smoothness (lower = sharper, higher = smoother)
)

# boundary line
plt.axvline(0, color="black", linestyle="--", linewidth=1)

plt.xlabel("Distance to Islet Boundary (µm)")
plt.ylabel("Relative cell density")
plt.title("Spatial distribution near islet boundary")

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
out = "/Users/kmeneses/Desktop/Figure_2_plots/SI/isletcellsonlykde_boundary_distribution"

plt.savefig(out + ".png", dpi=600, bbox_inches="tight")
plt.savefig(out + ".svg", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"

cell_types = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts"
    ]

# =========================
# DATA
# =========================
df = islet_adata_vasc.obs[
    [DIST_COL, CELLTYPE_COL]
].dropna().copy()

df[CELLTYPE_COL] = (
    df[CELLTYPE_COL]
    .astype(str)
    .str.strip()
)

# restrict window
df = df[
    np.abs(df[DIST_COL]) <= 50
]

df = df[
    df[CELLTYPE_COL].isin(cell_types)
]

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b"
    }

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,2))

sns.histplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    palette=palette,
    stat="probability",   # true fraction
    common_norm=False,
    bins=20,
    element="step",
    fill=False,
    linewidth=1.2
)

# boundary
plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

# labels
plt.xlabel("Distance to Islet Boundary (µm)")
plt.ylabel("Cell fraction")
plt.title("Spatial distribution near islet boundary")

sns.despine()

plt.tight_layout()

# =========================
# SAVE
# =========================
out = (
    "/Users/kmeneses/Desktop/"
    "Figure_2_plots/SI/"
    "isletcellsonly_hist_boundary_distribution"
)

plt.savefig(
    out + ".png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    out + ".svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    out + ".pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"

cell_types = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts"
    ]

# =========================
# DATA
# =========================
df = islet_adata_vasc.obs[
    [DIST_COL, CELLTYPE_COL]
].dropna().copy()

df[CELLTYPE_COL] = (
    df[CELLTYPE_COL]
    .astype(str)
    .str.strip()
)

# restrict window
df = df[
    np.abs(df[DIST_COL]) <= 50
]

df = df[
    df[CELLTYPE_COL].isin(cell_types)
]

# =========================
# COLORS
# =========================
palette = {
    "Pericytes": "#FD15FA",
    "Islet-associated Fibroblasts": "#f9a942",
    "Endothelial Cells": "#08c93b"
    }

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,2))

sns.histplot(
    data=df,
    x=DIST_COL,
    hue=CELLTYPE_COL,
    palette=palette,
    stat="probability",   # true fraction
    common_norm=False,
    bins=40,
    element="step",
    fill=False,
    linewidth=1.8
)

# boundary
plt.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1
)

# labels
plt.xlabel("Distance to Islet Boundary (µm)")
plt.ylabel("Cell fraction")
plt.title("Spatial distribution near islet boundary")

sns.despine()

plt.tight_layout()

# =========================
# SAVE
# =========================
out = (
    "/Users/kmeneses/Desktop/"
    "Figure_2_plots/SI/"
    "isletcellsonly_hist_boundary_distribution"
)

plt.savefig(
    out + ".png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    out + ".svg",
    bbox_inches="tight"
)

plt.savefig(
    out + ".pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"
LOCATION_COL = "Location"
SAMPLE_COL = "Sample"

cell_types_target = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts", 
    "Fibroblasts"
]

THRESH = 5  # µm

# =========================
# LOAD + CLEAN
# =========================
df = vasc_healthy.obs[[DIST_COL, CELLTYPE_COL, LOCATION_COL, SAMPLE_COL]].copy()

# clean labels (important)
df[CELLTYPE_COL] = df[CELLTYPE_COL].astype(str).str.strip()
df[LOCATION_COL] = df[LOCATION_COL].astype(str).str.strip()

# drop missing
df = df.dropna()

# =========================
# SPATIAL FILTER (🔥 better than strict Islet)
# =========================
# keep region around islet boundary instead of strict Location
df = df[np.abs(df[DIST_COL]) <= 50]

# keep only relevant cell types
df = df[df[CELLTYPE_COL].isin(cell_types_target)]

# =========================
# COMPUTE METRIC
# =========================
df["near_boundary"] = np.abs(df[DIST_COL]) <= THRESH

# =========================
# DONOR-LEVEL SUMMARY
# =========================
donor_summary = (
    df.groupby([SAMPLE_COL, CELLTYPE_COL])["near_boundary"]
    .mean()
    .reset_index()
)

donor_summary["percent"] = donor_summary["near_boundary"] * 100

# =========================
# CHECK GROUPS
# =========================
print("\nGroups present:")
print(donor_summary[CELLTYPE_COL].value_counts())

# =========================
# SAFE ORDER (🔥 prevents crash)
# =========================
order = [
    ct for ct in cell_types_target
    if ct in donor_summary[CELLTYPE_COL].unique()
]

print("\nPlot order:", order)

# =========================
# EXPORT (for Prism)
# =========================
donor_summary.to_csv(
    "/Users/kmeneses/Desktop/Figure_2_plots/SI/boundary_percent_donor.csv",
    index=False
)

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,3))

sns.boxplot(
    data=donor_summary,
    x=CELLTYPE_COL,
    y="percent",
    order=order,
    showfliers=False,
    color="white"
)

sns.stripplot(
    data=donor_summary,
    x=CELLTYPE_COL,
    y="percent",
    order=order,
    color="black",
    size=4,
    jitter=0.2
)

plt.ylabel(f"% cells within ±{THRESH} µm")
plt.xlabel("")
plt.title("Islet boundary enrichment")

plt.xticks(rotation=20)

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/boundary_percent_plot"

plt.savefig(out_base + ".png", dpi=600, bbox_inches="tight")
plt.savefig(out_base + ".svg", bbox_inches="tight")

plt.show()

print("\n✔ DONE — boundary enrichment computed and plotted")

In [ ]:
# SECTION 8 — SPATIAL ENRICHMENT ANALYSIS
# % cells within ±5 µm of islet boundary per donor per cell type
# Spatial window: ±50 µm around islet boundary
# KDE and step histogram plots (isletcells only vs all vascular)
# Donor-level Wilcoxon tests comparing proximity of vascular cell types
# Pericyte coverage: % ECs within 5 µm of a pericyte
# Distance bins (0–5, 5–10, 10–20, 20–40 µm) for pericytes and fibroblasts
# All results exported as CSV for GraphPad Prism

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"
LOCATION_COL = "Location"
SAMPLE_COL = "Sample"

cell_types_target = [
    "Endothelial Cells",
    "Pericytes",
    "Fibroblasts",
    "Islet-associated Fibroblasts"
]

THRESH = 5  # µm

# =========================
# LOAD + CLEAN
# =========================
df = vasc_healthy.obs[[DIST_COL, CELLTYPE_COL, LOCATION_COL, SAMPLE_COL]].copy()

# clean labels (important)
df[CELLTYPE_COL] = df[CELLTYPE_COL].astype(str).str.strip()
df[LOCATION_COL] = df[LOCATION_COL].astype(str).str.strip()

# drop missing
df = df.dropna()

# =========================
# SPATIAL FILTER (🔥 better than strict Islet)
# =========================
# keep region around islet boundary instead of strict Location
df = df[np.abs(df[DIST_COL]) <= 50]

# keep only relevant cell types
df = df[df[CELLTYPE_COL].isin(cell_types_target)]

# =========================
# COMPUTE METRIC
# =========================
df["near_boundary"] = np.abs(df[DIST_COL]) <= THRESH

# =========================
# DONOR-LEVEL SUMMARY
# =========================
donor_summary = (
    df.groupby([SAMPLE_COL, CELLTYPE_COL])["near_boundary"]
    .mean()
    .reset_index()
)

donor_summary["percent"] = donor_summary["near_boundary"] * 100

# =========================
# CHECK GROUPS
# =========================
print("\nGroups present:")
print(donor_summary[CELLTYPE_COL].value_counts())

# =========================
# SAFE ORDER (🔥 prevents crash)
# =========================
order = [
    ct for ct in cell_types_target
    if ct in donor_summary[CELLTYPE_COL].unique()
]

print("\nPlot order:", order)

# =========================
# EXPORT (for Prism)
# =========================
donor_summary.to_csv(
    "/Users/kmeneses/Desktop/Figure4_plots/SI/boundary_percent_donor.csv",
    index=False
)

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,3))

sns.boxplot(
    data=donor_summary,
    x=CELLTYPE_COL,
    y="percent",
    order=order,
    showfliers=False,
    color="white"
)

sns.stripplot(
    data=donor_summary,
    x=CELLTYPE_COL,
    y="percent",
    order=order,
    color="black",
    size=4,
    jitter=0.2
)

plt.ylabel(f"% cells within ±{THRESH} µm")
plt.xlabel("")
plt.title("Islet boundary enrichment")

plt.xticks(rotation=20)

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/boundary_percent_plot"

plt.savefig(out_base + ".png", dpi=600, bbox_inches="tight")
plt.savefig(out_base + ".svg", dpi=600, bbox_inches="tight")

plt.show()

print("\n✔ DONE — boundary enrichment computed and plotted")

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# SETTINGS
# =========================
DIST_COL = "dist_edge_islet_um"
CELLTYPE_COL = "cell_type_final"
LOCATION_COL = "Location"
SAMPLE_COL = "Sample"

cell_types_target = [
    "Endothelial Cells",
    "Pericytes",
    "Islet-associated Fibroblasts"
    ]

THRESH = 5  # µm

# =========================
# LOAD + CLEAN
# =========================
df = islet_adata_vasc.obs[[DIST_COL, CELLTYPE_COL, LOCATION_COL, SAMPLE_COL]].copy()

# clean labels (important)
df[CELLTYPE_COL] = df[CELLTYPE_COL].astype(str).str.strip()
df[LOCATION_COL] = df[LOCATION_COL].astype(str).str.strip()

# drop missing
df = df.dropna()

# =========================
# SPATIAL FILTER (🔥 better than strict Islet)
# =========================
# keep region around islet boundary instead of strict Location
df = df[np.abs(df[DIST_COL]) <= 50]

# keep only relevant cell types
df = df[df[CELLTYPE_COL].isin(cell_types_target)]

# =========================
# COMPUTE METRIC
# =========================
df["near_boundary"] = np.abs(df[DIST_COL]) <= THRESH

# =========================
# DONOR-LEVEL SUMMARY
# =========================
donor_summary = (
    df.groupby([SAMPLE_COL, CELLTYPE_COL])["near_boundary"]
    .mean()
    .reset_index()
)

donor_summary["percent"] = donor_summary["near_boundary"] * 100

# =========================
# CHECK GROUPS
# =========================
print("\nGroups present:")
print(donor_summary[CELLTYPE_COL].value_counts())

# =========================
# SAFE ORDER (🔥 prevents crash)
# =========================
order = [
    ct for ct in cell_types_target
    if ct in donor_summary[CELLTYPE_COL].unique()
]

print("\nPlot order:", order)

# =========================
# EXPORT (for Prism)
# =========================
donor_summary.to_csv(
    "/Users/kmeneses/Desktop/Figure_2_plots/SI/boundary_percent_donorISL.csv",
    index=False
)

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,3))

sns.boxplot(
    data=donor_summary,
    x=CELLTYPE_COL,
    y="percent",
    order=order,
    showfliers=False,
    color="white"
)

sns.stripplot(
    data=donor_summary,
    x=CELLTYPE_COL,
    y="percent",
    order=order,
    color="black",
    size=4,
    jitter=0.2
)

plt.ylabel(f"% cells within ±{THRESH} µm")
plt.xlabel("")
plt.title("Islet boundary enrichment")

plt.xticks(rotation=20)

sns.despine()
plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/boundary_percent_plotISL"

plt.savefig(out_base + ".png", dpi=600, bbox_inches="tight")
plt.savefig(out_base + ".svg", bbox_inches="tight")

plt.show()

print("\n✔ DONE — boundary enrichment computed and plotted")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================
# PARAMETERS
# =========================
CELLTYPE_COL = "cell_type_final"
HEALTH_COL = "Health_Status"
LOCATION_COL = "Location"
DIST_COL = "dist_edge_endo_um"   # 🔥 distance from each cell → nearest endothelial cell
SAMPLE_COL = "Sample"

THRESH = 2  # µm

# =========================
# SUBSET: Healthy Islet only
# =========================


# =========================
# PERICYTES ONLY
# =========================
peri = df[df[CELLTYPE_COL] == "Pericytes"].copy()

# drop missing values
peri = peri.dropna(subset=[DIST_COL])

# =========================
# COMPUTE COVERAGE
# =========================
peri["covered"] = peri[DIST_COL] <= THRESH

coverage_fraction = peri["covered"].mean()
print(f"Pericytes within {THRESH} µm of EC: {coverage_fraction:.3f}")

# =========================
# DONOR-LEVEL COVERAGE
# =========================
donor_cov = (
    peri.groupby(SAMPLE_COL)["covered"]
    .mean()
    .reset_index(name="coverage")
)

print("\nDonor-level coverage:")
print(donor_cov)

# =========================
# PLOT
# =========================
plt.figure(figsize=(3,3))

sns.boxplot(
    data=donor_cov,
    y="coverage",
    color="lightgray",
    width=0.5
)

sns.stripplot(
    data=donor_cov,
    y="coverage",
    color="black",
    size=5
)

plt.ylabel("Pericytes within 2 µm of EC (fraction)")
plt.xlabel("")
plt.title("Pericyte–endothelial proximity (Healthy Islet)")

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
sns.violinplot(
    data=donor_cov,
    y="coverage",
    color="lightgray",
    inner=None
)

sns.stripplot(
    data=donor_cov,
    y="coverage",
    color="black",
    size=5
)

In [ ]:
thresholds = [1, 2, 5, 10, 20]

results = []

for t in thresholds:
    peri[f"covered_{t}"] = peri["dist_edge_endo_um"] <= t
    cov = peri.groupby("Sample")[f"covered_{t}"].mean().mean()
    results.append((t, cov))

df_curve = pd.DataFrame(results, columns=["threshold", "coverage"])
for sample, d in peri.groupby("Sample"):
    vals = []
    for t in thresholds:
        vals.append((d["dist_edge_endo_um"] <= t).mean())
    plt.plot(thresholds, vals, alpha=0.3, color="gray")

# overlay mean
plt.plot(thresholds, df_curve["coverage"], color="black", linewidth=2)

sns.lineplot(data=df_curve, x="threshold", y="coverage", marker="o")

In [ ]:
df_curve

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import matplotlib as mpl

# -----------------------------
# Illustrator-friendly settings
# -----------------------------
mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# -----------------------------
# COLUMN NAMES
# -----------------------------
CELLTYPE_COL = "cell_type_final"
LOCATION_COL = "Location"
HEALTH_COL = "Health_Status"
DIST_COL = "dist_edge_endo_um"

# -----------------------------
# SUBSET: ISLET PERICYTES
# -----------------------------
peri = islet_adata_vasc[
    (islet_adata_vasc.obs[CELLTYPE_COL] == "Pericytes") &
    (islet_adata_vasc.obs[LOCATION_COL] == "Islet") &
    (~islet_adata_vasc.obs[DIST_COL].isna())
].copy()

df = peri.obs[[HEALTH_COL, "Sample", DIST_COL]].copy()

# -----------------------------
# BIN DISTANCES
# -----------------------------
bins = [0, 5, 10, 20, 40]
labels = ["0–5", "5–10", "10–20", "20–40"]

df["dist_bin"] = pd.cut(df[DIST_COL], bins=bins, labels=labels, include_lowest=True)

# -----------------------------
# COUNT + FRACTION
# -----------------------------
counts = df.groupby(["Sample", HEALTH_COL, "dist_bin"]).size().reset_index(name="count")

counts["fraction"] = counts["count"] / counts.groupby(["Sample", HEALTH_COL])["count"].transform("sum")

# -----------------------------
# PLOT
# -----------------------------
plt.figure(figsize=(6,4))

sns.barplot(
    data=counts,
    x="dist_bin",
    y="fraction",
    hue=HEALTH_COL
)

plt.xlabel("Distance to endothelial cells (µm)")
plt.ylabel("Fraction of pericytes")
plt.title("Pericyte distribution relative to endothelium")

plt.tight_layout()

# -----------------------------
# SAVE (SVG + PNG)
# -----------------------------
out_path = "/Volumes/Samsung_4TB/Figures_03062026/pericyte_distance_bins"

os.makedirs(os.path.dirname(out_path), exist_ok=True)

plt.savefig(f"{out_path}.svg", format="svg", bbox_inches="tight")
plt.savefig(f"{out_path}.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import matplotlib as mpl

# -----------------------------
# Illustrator-friendly settings
# -----------------------------
mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# -----------------------------
# COLUMN NAMES
# -----------------------------
CELLTYPE_COL = "cell_type_final"
HEALTH_COL = "Health_Status"
DIST_COL = "dist_edge_islet_um"

# -----------------------------
# SUBSET: ISLET-ASSOCIATED FIBROBLASTS
# -----------------------------
fib = islet_adata_vasc[
    (islet_adata_vasc.obs[CELLTYPE_COL] == "Islet-associated Fibroblasts") &
    (~islet_adata_vasc.obs[DIST_COL].isna())
].copy()

df = fib.obs[[HEALTH_COL, "Sample", DIST_COL]].copy()

# -----------------------------
# USE ABSOLUTE DISTANCE
# -----------------------------
df["abs_dist"] = df[DIST_COL].abs()

# -----------------------------
# BIN DISTANCES
# -----------------------------
bins = [0, 10, 25, 50, 100]
labels = ["0–10", "10–25", "25–50", "50–100"]

df["dist_bin"] = pd.cut(
    df["abs_dist"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# -----------------------------
# COUNT + FRACTION
# -----------------------------
counts = (
    df.groupby(["Sample", HEALTH_COL, "dist_bin"])
    .size()
    .reset_index(name="count")
)

counts["fraction"] = (
    counts["count"] /
    counts.groupby(["Sample", HEALTH_COL])["count"].transform("sum")
)

# -----------------------------
# PLOT
# -----------------------------
plt.figure(figsize=(6,4))

sns.barplot(
    data=counts,
    x="dist_bin",
    y="fraction",
    hue=HEALTH_COL
)

plt.xlabel("Distance to islet boundary (µm)")
plt.ylabel("Fraction of fibroblasts")
plt.title("Islet-associated fibroblast distance distribution")

plt.tight_layout()

# -----------------------------
# SAVE
# -----------------------------
out_path = "/Volumes/Samsung_4TB/Figures_03062026/islet_fibro_distance_bins"

os.makedirs(os.path.dirname(out_path), exist_ok=True)

plt.savefig(f"{out_path}.svg", format="svg", bbox_inches="tight")
plt.savefig(f"{out_path}.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import matplotlib as mpl

# -----------------------------
# Illustrator-friendly settings
# -----------------------------
mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# -----------------------------
# COLUMN NAMES
# -----------------------------
CELLTYPE_COL = "cell_type_final"
HEALTH_COL = "Health_Status"
DIST_COL = "dist_edge_islet_um"

# -----------------------------
# SUBSET: ISLET-ASSOCIATED FIBROBLASTS
# -----------------------------
fib = islet_adata_vasc[
    (islet_adata_vasc.obs[CELLTYPE_COL] == "Pericytes") &
    (~islet_adata_vasc.obs[DIST_COL].isna())
].copy()

df = fib.obs[[HEALTH_COL, "Sample", DIST_COL]].copy()

# -----------------------------
# USE ABSOLUTE DISTANCE
# -----------------------------
df["abs_dist"] = df[DIST_COL].abs()

# -----------------------------
# BIN DISTANCES
# -----------------------------
bins = [0, 10, 25, 50, 100]
labels = ["0–10", "10–25", "25–50", "50–100"]

df["dist_bin"] = pd.cut(
    df["abs_dist"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# -----------------------------
# COUNT + FRACTION
# -----------------------------
counts = (
    df.groupby(["Sample", HEALTH_COL, "dist_bin"])
    .size()
    .reset_index(name="count")
)

counts["fraction"] = (
    counts["count"] /
    counts.groupby(["Sample", HEALTH_COL])["count"].transform("sum")
)

# -----------------------------
# PLOT
# -----------------------------
plt.figure(figsize=(6,4))

sns.barplot(
    data=counts,
    x="dist_bin",
    y="fraction",
    hue=HEALTH_COL
)

plt.xlabel("Distance to islet boundary (µm)")
plt.ylabel("Fraction of fibroblasts")
plt.title("Islet-associated fibroblast distance distribution")

plt.tight_layout()

# -----------------------------
# SAVE
# -----------------------------
out_path = "/Volumes/Samsung_4TB/Figures_03062026/islet_peri_distance_bins"

os.makedirs(os.path.dirname(out_path), exist_ok=True)

plt.savefig(f"{out_path}.svg", format="svg", bbox_inches="tight")
plt.savefig(f"{out_path}.png", dpi=300, bbox_inches="tight")

plt.show()

# SECTION: DIFFERENTIAL GENE EXPRESSION + CROSS-DATASET VALIDATION (Craig–Shapiro)
 Identifies transcriptional programs that distinguish
 vascular cell types and validates findings in an
 independent reference dataset (adata_new_ref /
  Craig–Shapiro cohort).
#
# Comparisons run (Wilcoxon, use_raw=False):
  1. Pericytes vs Endothelial Cells       (islet only)
   2. Pericytes vs Islet-assoc Fibroblasts (islet only)
   3. Islet-assoc Fibroblasts vs Pericytes (islet only)
   4. Pericytes: Islet vs Exocrine         (location DEG)

# Cross-dataset check:
   - Same comparisons repeated in adata_new_ref (Location=Islet, or both for pericyte comparison) 
     (cell_type_merged: Pericyte / Endothelial Cells /
      Fibroblasts) to confirm reproducibility
   - Overlapping genes filtered to 300-gene spatial panel
     (islet_adata_vasc.var_names)

# Outputs:
   - Full DEG tables (CSV)
   - logFC heatmaps for curated ECM / BM gene sets
   - Volcano plots (Islet vs Exocrine pericytes)
   - Lollipop / barplots of selected niche-associated genes
   - ECM / BM module scores on UMAP (BM_score, ECM_score)
   - Paired donor-level score summaries → Wilcoxon test

 Key objects: pericytesf, islet_adata_vasc, adata_new_ref

In [ ]:
import scanpy as sc
import pandas as pd
import os

# =========================
# SETTINGS
# =========================
CELLTYPE = "cell_type_final"
HEALTH = "Health_Status"
LOCATION = "Location"

OUT_DIR = "/Users/kmeneses/Desktop/Fig_plots/islet_DEG_compaere"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# SUBSET: Healthy Islet
# =========================


# =========================
# DE 1: Pericytes vs Endothelial Cells
# =========================
ad_endo = islet_adata_vasc[islet_adata_vasc.obs[CELLTYPE].isin(["Pericytes","Endothelial Cells"])].copy()

sc.tl.rank_genes_groups(
    ad_endo,
    groupby=CELLTYPE,
    groups=["Pericytes"],
    reference="Endothelial Cells",
    method="wilcoxon",
    use_raw=False
)

deg_pc_vs_endo = sc.get.rank_genes_groups_df(ad_endo, group="Pericytes")

# Clean column names
deg_pc_vs_endo = deg_pc_vs_endo.rename(columns={
    "names": "gene",
    "logfoldchanges": "logFC",
    "pvals_adj": "FDR"
})

# =========================
# DE 2: Pericytes vs Peri-islet Fibroblasts
# =========================
ad_fib = islet_adata_vasc[islet_adata_vasc.obs[CELLTYPE].isin(["Pericytes","Islet-associated Fibroblasts"])].copy()

sc.tl.rank_genes_groups(
    ad_fib,
    groupby=CELLTYPE,
    groups=["Pericytes"],
    reference="Islet-associated Fibroblasts",
    method="wilcoxon",
    use_raw=False
)

deg_pc_vs_fib = sc.get.rank_genes_groups_df(ad_fib, group="Pericytes")

deg_pc_vs_fib = deg_pc_vs_fib.rename(columns={
    "names": "gene",
    "logfoldchanges": "logFC",
    "pvals_adj": "FDR"
})

# =========================
# OPTIONAL: sort by effect size
# =========================
deg_pc_vs_endo = deg_pc_vs_endo.sort_values("logFC", ascending=False)
deg_pc_vs_fib = deg_pc_vs_fib.sort_values("logFC", ascending=False)

# =========================
# EXPORT FULL TABLES
# =========================
deg_pc_vs_endo.to_csv(
    f"{OUT_DIR}/Pericytes_vs_Endothelial_ALL_GErNES.csv",
    index=False
)

deg_pc_vs_fib.to_csv(
    f"{OUT_DIR}/Pericytes_vs_IsletFibro_ALL_GrENES.csv",
    index=False
)

print("✔ DONE — full DEG tables exported")

In [ ]:
deg_pc_vs_endo.head(50)

In [ ]:
deg_pc_vs_fib.head(50)

In [ ]:
ad_periusle = islet_adata_vasc[islet_adata_vasc.obs[CELLTYPE].isin(["Islet-associated Fibroblasts","Pericytes"])].copy()

sc.tl.rank_genes_groups(
    ad_periusle,
    groupby=CELLTYPE,
    groups=["Islet-associated Fibroblasts"],
    reference="Pericytes",
    method="wilcoxon",
    use_raw=False
)

deg_isl_vs_peri = sc.get.rank_genes_groups_df(ad_periusle, group="Islet-associated Fibroblasts")

# Clean column names
deg_isl_vs_peri = deg_isl_vs_peri.rename(columns={
    "names": "gene",
    "logfoldchanges": "logFC",
    "pvals_adj": "FDR"
})

In [ ]:
deg_isl_vs_peri.head(50)

In [ ]:
# fibroblast-enriched = negative logFC
fib_enriched = deg_pc_vs_fib[
    deg_pc_vs_fib["logFC"] < 0
].copy()

# sort strongest enrichment
fib_enriched = fib_enriched.sort_values("logFC")

# optional: filter for significance
fib_enriched = fib_enriched[fib_enriched["FDR"] < 0.05]

# top genes
fib_enriched.head(20)

In [ ]:
# endothelial-enriched = negative logFC
endo_enriched = deg_pc_vs_endo[
    deg_pc_vs_endo["logFC"] < 0
].copy()

# sort strongest enrichment
endo_enriched = endo_enriched.sort_values("logFC")

# optional: significance filter
endo_enriched = endo_enriched[endo_enriched["FDR"] < 0.05]

# preview
endo_enriched.head(20)

In [ ]:
import pandas as pd
import os

CELLTYPE_COL = "cell_type"
LOCATION_COL = "Location"
HEALTH_COL = "Health_Status"
DONOR_COL = "Sample"

vasc_types = [
    "Pericytes",
    "Endothelial Cells",
    "Islet-associated Fibroblasts",
    "Fibroblasts"
]

out_dir = "/Users/kmeneses/Desktop/prism_exports"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------
# subset vascular cells
# ---------------------------------------------------

df = vasc_healthy.obs.copy()
df = df[df[CELLTYPE_COL].isin(vasc_types)].copy()

# ---------------------------------------------------
# count cells
# ---------------------------------------------------

counts = (
    df
    .groupby([DONOR_COL, HEALTH_COL, LOCATION_COL, CELLTYPE_COL])
    .size()
    .reset_index(name="n_cells")
)

totals = (
    counts
    .groupby([DONOR_COL, HEALTH_COL, LOCATION_COL])["n_cells"]
    .sum()
    .reset_index(name="total_cells")
)

counts = counts.merge(totals)
counts["proportion"] = counts["n_cells"] / counts["total_cells"]

# ---------------------------------------------------
# function to remove biologically impossible cells
# ---------------------------------------------------

def filter_compartment(data, compartment):

    data = data[data[LOCATION_COL] == compartment].copy()

    if compartment == "Islet":
        data = data[data[CELLTYPE_COL] != "Fibroblasts"]

    if compartment == "Exocrine":
        data = data[data[CELLTYPE_COL] != "Islet-associated Fibroblasts"]

    return data


# ---------------------------------------------------
# EXPORT CELL TYPE PROPORTIONS
# ---------------------------------------------------

for compartment in ["Islet", "Exocrine"]:

    comp_df = filter_compartment(counts, compartment)

    for ct in comp_df[CELLTYPE_COL].unique():

        ct_df = comp_df[comp_df[CELLTYPE_COL] == ct]

        prism = ct_df.pivot(
            index=DONOR_COL,
            columns=HEALTH_COL,
            values="proportion"
        )

        prism.to_csv(
            f"{out_dir}/{compartment}_{ct}_proportions.csv"
        )


# ---------------------------------------------------
# EXPORT PERICYTE : ENDOTHELIAL RATIO
# ---------------------------------------------------

ratio_df = counts[
    counts[CELLTYPE_COL].isin(["Pericytes","Endothelial Cells"])
]

ratio_pivot = ratio_df.pivot_table(
    index=[DONOR_COL, HEALTH_COL, LOCATION_COL],
    columns=CELLTYPE_COL,
    values="n_cells",
    aggfunc="sum"
)

ratio_pivot["Pericyte_Endothelial_Ratio"] = (
    ratio_pivot["Pericytes"] /
    ratio_pivot["Endothelial Cells"]
)

ratio_pivot.reset_index(inplace=True)

for compartment in ["Islet","Exocrine"]:

    comp = ratio_pivot[ratio_pivot[LOCATION_COL] == compartment]

    prism = comp.pivot(
        index=DONOR_COL,
        columns=HEALTH_COL,
        values="Pericyte_Endothelial_Ratio"
    )

    prism.to_csv(
        f"{out_dir}/{compartment}_Pericyte_Endothelial_ratio.csv"
    )


# ---------------------------------------------------
# EXPORT PERICYTE FRACTION
# ---------------------------------------------------

peri_df = counts[counts[CELLTYPE_COL] == "Pericytes"]

for compartment in ["Islet","Exocrine"]:

    comp = peri_df[peri_df[LOCATION_COL] == compartment]

    prism = comp.pivot(
        index=DONOR_COL,
        columns=HEALTH_COL,
        values="proportion"
    )

    prism.to_csv(
        f"{out_dir}/{compartment}_Pericyte_fraction.csv"
    )


print("All Prism tables exported to:", out_dir)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
import os

# =========================
# SETTINGS
# =========================
genes_plot = [
    # pericyte markers
    "TINAGL1", "RGS5", "FLT1", "COL6A1", "COL6A2",
     "COL1A2", "COL1A1",
    
    # BM / shared
    "LAMA2"
    
    # endothelial markers
    
]
OUT = "/Users/kmeneses/Desktop/BM_logFC_heatmap_fib"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

# =========================
# GET logFC from DEG
# =========================
df = deg_pc_vs_fib.set_index("gene").loc[genes_plot]

# Pericytes vs Endo → logFC
# positive = pericyte enriched
# negative = endothelial enriched

df_plot = pd.DataFrame({
    "Pericytes vs Islet-assocaited Fibroblasts": df["logFC"]
})

# =========================
# PLOT
# =========================
plt.figure(figsize=(1,2.5))

sns.heatmap(
    df_plot,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "logFC"}
)

plt.title("Differential Expression")
plt.ylabel("")
plt.xlabel("")

plt.tight_layout()

# =========================
# SAVE
# =========================
plt.savefig(f"{OUT}.svg")
plt.savefig(f"{OUT}.png", dpi=600)

plt.show()

In [ ]:
adata_new_ref

In [ ]:
import pandas as pd
import os

CELLTYPE_COL = "cell_type_merged"
LOCATION_COL = "Location"

DONOR_COL = "donor_id"

vasc_types = [
    "Pericyte",
    "Endothelial Cells",
    "Fibroblasts"
]

out_dir = "/Users/kmeneses/Desktop/prism_exports"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------
# subset vascular cells
# ---------------------------------------------------

df = adata_new_ref.obs.copy()
df = df[df[CELLTYPE_COL].isin(vasc_types)].copy()

# ---------------------------------------------------
# count cells
# ---------------------------------------------------

counts = (
    df
    .groupby([DONOR_COL, HEALTH_COL, LOCATION_COL, CELLTYPE_COL])
    .size()
    .reset_index(name="n_cells")
)

totals = (
    counts
    .groupby([DONOR_COL, HEALTH_COL, LOCATION_COL])["n_cells"]
    .sum()
    .reset_index(name="total_cells")
)

counts = counts.merge(totals)
counts["proportion"] = counts["n_cells"] / counts["total_cells"]

# ---------------------------------------------------
# function to remove biologically impossible cells
# ---------------------------------------------------

def filter_compartment(data, compartment):

    data = data[data[LOCATION_COL] == compartment].copy()

    if compartment == "Islet":
        data = data[data[CELLTYPE_COL] != "Fibroblasts"]


    return data


# ---------------------------------------------------
# EXPORT CELL TYPE PROPORTIONS
# ---------------------------------------------------

for compartment in ["Islet", "Exocrine"]:

    comp_df = filter_compartment(counts, compartment)

    for ct in comp_df[CELLTYPE_COL].unique():

        ct_df = comp_df[comp_df[CELLTYPE_COL] == ct]

        prism = ct_df.pivot(
            index=DONOR_COL,
            columns=HEALTH_COL,
            values="proportion"
        )

        prism.to_csv(
            f"{out_dir}/{compartment}_{ct}_proportions.csv"
        )


# ---------------------------------------------------
# EXPORT PERICYTE : ENDOTHELIAL RATIO
# ---------------------------------------------------

ratio_df = counts[
    counts[CELLTYPE_COL].isin(["Pericyte","Endothelial Cells"])
]

ratio_pivot = ratio_df.pivot_table(
    index=[DONOR_COL, HEALTH_COL, LOCATION_COL],
    columns=CELLTYPE_COL,
    values="n_cells",
    aggfunc="sum"
)

ratio_pivot["Pericyte_Endothelial_Ratio"] = (
    ratio_pivot["Pericyte"] /
    ratio_pivot["Endothelial Cells"]
)

ratio_pivot.reset_index(inplace=True)

for compartment in ["Islet","Exocrine"]:

    comp = ratio_pivot[ratio_pivot[LOCATION_COL] == compartment]

    prism = comp.pivot(
        index=DONOR_COL,
        columns=HEALTH_COL,
        values="Pericyte_Endothelial_Ratio"
    )

    prism.to_csv(
        f"{out_dir}/{compartment}_Pericyte_Endothelial_ratio.csv"
    )


# ---------------------------------------------------
# EXPORT PERICYTE FRACTION
# ---------------------------------------------------

peri_df = counts[counts[CELLTYPE_COL] == "Pericytes"]

for compartment in ["Islet","Exocrine"]:

    comp = peri_df[peri_df[LOCATION_COL] == compartment]

    prism = comp.pivot(
        index=DONOR_COL,
        columns=HEALTH_COL,
        values="proportion"
    )

    prism.to_csv(
        f"{out_dir}/{compartment}_Pericyte_fraction.csv"
    )


print("All Prism tables exported to:", out_dir)

In [ ]:
deg_pc_vs_endo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
import os

# =========================
# SETTINGS
# =========================
genes_plot = [
    # pericyte markers
    "COL5A3", "MFGE8", "COL3A1", "MYL9", "SSTR2", "LAMA5",
    
    # BM / shared
    "KDR","HSPG2","SOX18"
]
OUT = "/Users/kmeneses/Desktop/BM_logFC_heatmap"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

# =========================
# GET logFC from DEG
# =========================
df = deg_pc_vs_endo.set_index("gene").loc[genes_plot]

# Pericytes vs Endo → logFC
# positive = pericyte enriched
# negative = endothelial enriched

df_plot = pd.DataFrame({
    "Pericytes vs Endo": df["logFC"]
})

# =========================
# PLOT
# =========================
plt.figure(figsize=(1,2.5))

sns.heatmap(
    df_plot,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "logFC"}
)

plt.title("Differential Expression")
plt.ylabel("")
plt.xlabel("")

plt.tight_layout()

# =========================
# SAVE
# =========================
plt.savefig(f"{OUT}.svg")
plt.savefig(f"{OUT}.png", dpi=600)

plt.show()

In [ ]:
NEW_REF_PATH = "/Volumes/Samsung_4TB/Desktop_copy/07092025/shp_vascells_concord_relabled_mergedECslabel.h5ad"
adata_new_ref = sc.read_h5ad(NEW_REF_PATH)

In [ ]:
adata_new_ref.obs['cell_type_merged']

In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# SUBSET
# =========================
# =========================
# SUBSET TO ISLET ONLY
# =========================
adata_deg = adata_new_ref[
    (adata_new_ref.obs["Location"] == "Islet") &
    (adata_new_ref.obs["cell_type_merged"].isin(
        ["Pericyte", "Endothelial Cells"]
    ))
].copy()

print(adata_deg)

print(
    adata_deg.obs["cell_type_merged"]
    .value_counts()
)

print(
    adata_deg.obs["Location"]
    .value_counts()
)
# =========================
# DEG
# =========================
sc.tl.rank_genes_groups(
    adata_deg,
    groupby="cell_type_merged",
    groups=["Pericyte"],
    reference="Endothelial Cells",
    method="wilcoxon",
    use_raw=False
)

deg = sc.get.rank_genes_groups_df(
    adata_deg,
    group="Pericyte"
)

# =========================
# GENES TO PLOT
# =========================
genes_plot = [
    "COL5A3",
    "MFGE8",
    "COL3A1",
    "MYL9",
    "SSTR2",
    "LAMA5",
    "KDR",
    "HSPG2",
    "SOX18"
]

# =========================
# PREPARE HEATMAP TABLE
# =========================
deg = deg.set_index("names")

genes_present = [
    g for g in genes_plot
    if g in deg.index
]

df_plot = pd.DataFrame({
    "Pericytes vs\nEndothelial Cells":
        deg.loc[
            genes_present,
            "logfoldchanges"
        ]
})

# optional order
df_plot = df_plot.loc[genes_present]

# =========================
# PLOT
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

plt.figure(figsize=(3, 2.8))

sns.heatmap(
    df_plot,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "log2FC"}
)

plt.title("Pericyte vs\nEndothelial Cells")
plt.xlabel("")
plt.ylabel("")

plt.tight_layout()

# =========================
# SAVE
# =========================
OUT = "/Users/kmeneses/Desktop/Pericyte_vs_EC_logFC_heatmap_CRSH"

plt.savefig(f"{OUT}.svg")
plt.savefig(f"{OUT}.pdf")
plt.savefig(f"{OUT}.png", dpi=600)

plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# SUBSET
# =========================
# =========================
# SUBSET TO ISLET ONLY
# =========================
adata_deg = adata_new_ref[
    (adata_new_ref.obs["Location"] == "Islet") &
    (adata_new_ref.obs["cell_type_merged"].isin(
        ["Pericyte", "Fibroblasts"]
    ))
].copy()

print(adata_deg)

print(
    adata_deg.obs["cell_type_merged"]
    .value_counts()
)

print(
    adata_deg.obs["Location"]
    .value_counts()
)
# =========================
# DEG
# =========================
sc.tl.rank_genes_groups(
    adata_deg,
    groupby="cell_type_merged",
    groups=["Pericyte"],
    reference="Fibroblasts",
    method="wilcoxon",
    use_raw=False
)

deg = sc.get.rank_genes_groups_df(
    adata_deg,
    group="Pericyte"
)

# =========================
# GENES TO PLOT
genes_plot = [
    # pericyte markers
    "TINAGL1", "RGS5", "FLT1", "COL6A1", "COL6A2",
     "COL1A2", "COL1A1",
    
    # BM / shared
    "LAMA2"
    
    # endothelial markers
    
]

# =========================
# PREPARE HEATMAP TABLE
# =========================
deg = deg.set_index("names")

genes_present = [
    g for g in genes_plot
    if g in deg.index
]

df_plot = pd.DataFrame({
    "Pericytes vs\Fibroblasts":
        deg.loc[
            genes_present,
            "logfoldchanges"
        ]
})

# optional order
df_plot = df_plot.loc[genes_present]

# =========================
# PLOT
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

plt.figure(figsize=(2.2, 2.8))

sns.heatmap(
    df_plot,
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "log2FC"}
)

plt.title("Pericyte vs\Fibroblasts")
plt.xlabel("")
plt.ylabel("")

plt.tight_layout()

# =========================
# SAVE
# =========================
OUT = "/Users/kmeneses/Desktop/Pericyte_vs_Fibroblasts_logFC_heatmap_CRSH"

plt.savefig(f"{OUT}.svg")
plt.savefig(f"{OUT}.pdf")
plt.savefig(f"{OUT}.png", dpi=600)

plt.show()

In [ ]:
adata_new_ref.obs['Location']

In [ ]:
import scanpy as sc
import pandas as pd

# =====================================================
# SETTINGS
# =====================================================

CELLTYPE_COL = "cell_type_merged"
LOCATION_COL = "Location"

GROUPS = [
    "Pericyte",
    "Endothelial Cells"
]

# =====================================================
# SUBSET:
# ISLET VASCULAR CELLS ONLY
# =====================================================

adata_deg = adata_new_ref[
    (adata_new_ref.obs[LOCATION_COL] == "Islet") &
    (adata_new_ref.obs[CELLTYPE_COL].isin(GROUPS))
].copy()

# =====================================================
# CHECK
# =====================================================

print(
    adata_deg.obs[CELLTYPE_COL]
    .value_counts()
)

# =====================================================
# RUN DGE
# =====================================================

sc.tl.rank_genes_groups(
    adata_deg,
    groupby=CELLTYPE_COL,
    groups=["Pericyte"],
    reference="Endothelial Cells",
    method="wilcoxon",
    use_raw=False
)

# =====================================================
# EXTRACT RESULTS
# =====================================================

deg_df = sc.get.rank_genes_groups_df(
    adata_deg,
    group="Pericyte"
)

# =====================================================
# FILTER SIGNIFICANT
# =====================================================

deg_sig = deg_df[
    (deg_df["pvals_adj"] < 0.05) &
    (abs(deg_df["logfoldchanges"]) > 0.25)
].copy()

# =====================================================
# SORT
# =====================================================

deg_sig = deg_sig.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# VIEW TOP GENES
# =====================================================

print(
    deg_sig.head(50)
)

# =====================================================
# SAVE
# =====================================================

deg_sig.to_csv(
    "/Users/kmeneses/Desktop/Islet_Pericytes_vs_Endothelial_DEG.csv",
    index=False
)

print(
    "\nSaved DEG table."
)

In [ ]:
# =====================================================
# GET ENDOTHELIAL-ENRICHED GENES
# =====================================================

deg_endo = deg_df[
    (deg_df["pvals_adj"] < 0.05) &
    (deg_df["logfoldchanges"] < -0.25)
].copy()

# =====================================================
# CONVERT TO POSITIVE FC
# easier to interpret
# =====================================================

deg_endo["Endo_logFC"] = (
    deg_endo["logfoldchanges"] * -1
)

# =====================================================
# SORT
# =====================================================

deg_endo = deg_endo.sort_values(
    "Endo_logFC",
    ascending=False
)

# =====================================================
# VIEW TOP GENES
# =====================================================

print(
    deg_endo.head(100)
)

# =====================================================
# SAVE
# =====================================================

deg_endo.to_csv(
    "/Users/kmeneses/Desktop/Endothelial_vs_Pericyte_DEG.csv",
    index=False
)

print(
    "\nSaved endothelial-enriched DEG table."
)

In [ ]:
deg_endo_panel = deg_endo[

    deg_endo["names"].isin(panel_genes)

].copy()

# =====================================================

# SORT

# =====================================================

deg_endo_panel = deg_endo_panel.sort_values(

    "Endo_logFC",

    ascending=False

)

# =====================================================

# VIEW

# =====================================================

print(

    "Filtered genes:",

    len(deg_endo_panel)

)

print(

    deg_endo_panel.head(100)

)

# =====================================================

# SAVE

# =====================================================

deg_endo_panel.to_csv(

    "/Users/kmeneses/Desktop/Endothelial_vs_Pericyte_300G_panel.csv",

    index=False

)

print(

    "\nSaved filtered endothelial DEG table."

)

In [ ]:
# =====================================================
# GET 300-GENE PANEL FROM ANNDATA
# =====================================================

panel_genes = (
    islet_adata_vasc.var_names
    .tolist()
)

# =====================================================
# CHECK
# =====================================================

print(
    "Number of panel genes:",
    len(panel_genes)
)

print(
    panel_genes[:20]
)

# =====================================================
# OPTIONAL:
# SAVE PANEL
# =====================================================

import pandas as pd

pd.DataFrame({
    "gene": panel_genes
}).to_csv(
    "/Users/kmeneses/Desktop/panel_genes.csv",
    index=False
)

print(
    "\nSaved panel gene list."
)

In [ ]:
# =====================================================
# FILTER DEG TO 300-GENE PANEL
# =====================================================

# your panel genes list
panel_genes = [
    # paste full 300-gene panel here
]

# =====================================================
# KEEP ONLY PANEL GENES
# =====================================================

deg_panel = deg_sig[
    deg_sig["names"].isin(panel_genes)
].copy()

# =====================================================
# SORT BY LOGFC
# =====================================================

deg_panel = deg_panel.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# VIEW
# =====================================================

print(deg_panel)

# =====================================================
# SAVE
# =====================================================

deg_panel.to_csv(
    "/Users/kmeneses/Desktop/Islet_Pericyte_vs_Endothelial_300G_DEG.csv",
    index=False
)

print(
    "\nSaved filtered DEG table."
)

In [ ]:
deg_sig["names"]

In [ ]:
print(islet_adata_vasc.var.head())

print(islet_adata_vasc.var.columns)

In [ ]:
# =====================================================
# PANEL GENES
# =====================================================

panel_genes = (
    islet_adata_vasc.var.index
    .astype(str)
    .str.strip()
    .tolist()
)

# =====================================================
# CLEAN DEG NAMES
# =====================================================

deg_sig["names"] = (
    deg_sig["names"]
    .astype(str)
    .str.strip()
)

# =====================================================
# CHECK OVERLAP
# =====================================================

overlap = (
    set(deg_sig["names"])
    &
    set(panel_genes)
)

print(
    "Overlap:",
    len(overlap)
)

print(
    list(overlap)[:20]
)

# =====================================================
# FILTER
# =====================================================

deg_panel = deg_sig[
    deg_sig["names"].isin(panel_genes)
].copy()

# =====================================================
# SORT
# =====================================================

deg_panel = deg_panel.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# VIEW
# =====================================================

print(deg_panel.head(50))

# =====================================================
# SAVE
# =====================================================

deg_panel.to_csv(
    "/Users/kmeneses/Desktop/Pericyte_vs_Endothelial_300G_panel.csv",
    index=False
)

print("\nSaved.")

In [ ]:
adata_new_ref.obs['cell_type_merged']

In [ ]:
import scanpy as sc
import pandas as pd

# =====================================================
# SETTINGS
# =====================================================

CELLTYPE_COL = "cell_type_merged"
LOCATION_COL = "Location"

GROUPS = [
    "Pericyte",
    "Fibroblasts"
]

# =====================================================
# SUBSET:
# ISLET MESENCHYMAL CELLS ONLY
# =====================================================

adata_deg = adata_new_ref[
    (adata_new_ref.obs[LOCATION_COL] == "Islet") &
    (adata_new_ref.obs[CELLTYPE_COL].isin(GROUPS))
].copy()

# =====================================================
# CHECK COUNTS
# =====================================================

print(
    adata_deg.obs[CELLTYPE_COL]
    .value_counts()
)

# =====================================================
# RUN DGE
# =====================================================

sc.tl.rank_genes_groups(
    adata_deg,
    groupby=CELLTYPE_COL,
    groups=["Pericyte"],
    reference="Fibroblasts",
    method="wilcoxon",
    use_raw=False
)

# =====================================================
# EXTRACT RESULTS
# =====================================================

deg_df = sc.get.rank_genes_groups_df(
    adata_deg,
    group="Pericyte"
)

# =====================================================
# FILTER SIGNIFICANT
# =====================================================

deg_sig = deg_df[
    (deg_df["pvals_adj"] < 0.05) &
    (deg_df["logfoldchanges"] > 1)
].copy()

# =====================================================
# GET 300-GENE PANEL
# =====================================================

panel_genes = (
    adata_new_ref.var.index
    .astype(str)
    .str.strip()
    .tolist()
)

# =====================================================
# CLEAN GENE NAMES
# =====================================================

deg_sig["names"] = (
    deg_sig["names"]
    .astype(str)
    .str.strip()
)

# =====================================================
# FILTER TO PANEL GENES
# =====================================================

deg_panel = deg_sig[
    deg_sig["names"].isin(panel_genes)
].copy()

# =====================================================
# REMOVE COMMON CONTAMINATION
# =====================================================



# =====================================================
# SORT
# =====================================================

deg_panel = deg_panel.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# VIEW TOP GENES
# =====================================================

print(
    deg_panel.head(100)
)

# =====================================================
# SAVE
# =====================================================

deg_panel.to_csv(
    "/Users/kmeneses/Desktop/Pericytes_vs_IsletNDFibroblasts_DEG.csv",
    index=False
)

print(
    "\nSaved DEG table."
)

In [ ]:
# =====================================================
# GET PANEL GENES
# =====================================================

panel_genes = (
    islet_adata_vasc.var.index
    .astype(str)
    .str.strip()
    .tolist()
)

# =====================================================
# CLEAN DEG GENE NAMES
# =====================================================

deg_fib["names"] = (
    deg_fib["names"]
    .astype(str)
    .str.strip()
)

# =====================================================
# FILTER TO PANEL GENES
# =====================================================

deg_fib_panel = deg_fib[
    deg_fib["names"].isin(panel_genes)
].copy()

# =====================================================
# SORT
# =====================================================

deg_fib_panel = deg_fib_panel.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# VIEW
# =====================================================

print(
    "Filtered genes:",
    len(deg_fib_panel)
)

print(
    deg_fib_panel.head(100)
)

# =====================================================
# SAVE
# =====================================================

deg_fib_panel.to_csv(
    "/Users/kmeneses/Desktop/Fibroblasts_vs_Pericyte_300G_panel.csv",
    index=False
)

print("\nSaved filtered DEG table.")

In [ ]:
# =====================================================
# GET PANEL GENES
# =====================================================

panel_genes = (
    islet_adata_vasc.var.index
    .astype(str)
    .str.strip()
    .tolist()
)

# =====================================================
# CLEAN DEG GENE NAMES
# =====================================================

deg_sig["names"] = (
    deg_sig["names"]
    .astype(str)
    .str.strip()
)

# =====================================================
# FILTER TO PANEL GENES
# =====================================================

deg_panel = deg_sig[
    deg_sig["names"].isin(panel_genes)
].copy()

# =====================================================
# SORT
# =====================================================

deg_panel = deg_panel.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# VIEW
# =====================================================

print(
    "Filtered genes:",
    len(deg_panel)
)

print(
    deg_panel.head(100)
)

# =====================================================
# SAVE
# =====================================================

deg_panel.to_csv(
    "/Users/kmeneses/Desktop/Pericytes_vs_IsletFibroblasts_300G_panel.csv",
    index=False
)

print("\nSaved filtered DEG table.")

In [ ]:
# =========================
# RUN DEG
# =========================
sc.tl.rank_genes_groups(
    adata_deg,
    groupby=CELLTYPE_COL,
    groups=["Fibroblasts"],
    reference="Pericyte",
    method="wilcoxon",
    use_raw=False
)

deg_fib = sc.get.rank_genes_groups_df(
    adata_deg,
    group="Fibroblasts"
)

# =========================
# FILTER
# =========================
deg_fib = deg_fib[
    (deg_fib["pvals_adj"] < 0.05) &
    (deg_fib["logfoldchanges"] > 1)
].copy()

deg_fib = deg_fib.sort_values(
    "logfoldchanges",
    ascending=False
)

# =========================
# SAVE CSV
# =========================
out_file = "/Users/kmeneses/Desktop/Fibroblasts_vs_Pericyte_DEG.csv"

deg_fib.to_csv(
    out_file,
    index=False
)

print(f"Saved: {out_file}")
print(deg_fib.head(100))

In [ ]:
print(deg_fib.head(100))

In [ ]:
# =====================================================
# ECM GENE LIST
# =====================================================

ecm_genes = panel_genes

# =====================================================
# FILTER
# =====================================================

deg_ecm = deg_fib[
    deg_fib["names"].isin(ecm_genes)
].copy()

# =====================================================
# SORT
# =====================================================

deg_ecm = deg_ecm.sort_values(
    "logfoldchanges",
    ascending=False
)

# =====================================================
# VIEW
# =====================================================

print(deg_ecm)

# =====================================================
# SAVE
# =====================================================

deg_ecm.to_csv(
    "/Users/kmeneses/Desktop/Fibroblast_ECM_DEG.csv",
    index=False
)

print("\nSaved ECM-filtered DEG table.")

In [ ]:
print(deg_ecm.head(60))

In [ ]:
vasc_healthy.obs['cell_type_final'].value_counts()

In [ ]:
pericytesf = vasc_healthy[
    vasc_healthy.obs["cell_type_final"] == "Pericytes"
].copy()


In [ ]:

sc.pp.normalize_total(pericytesf, target_sum=1e4)
sc.tl.rank_genes_groups(
    pericytesf,
    groupby="Location",
    groups=["Exocrine"],
    reference="Islet",
    method="wilcoxon", 
    use_raw=False,
    key_added='deg_exo'
)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
LOCATION_COL = "Location"

fc_thresh = 0.2
p_thresh = 0.05

# =========================
# SUBSET:
# HEALTHY PERICYTES

print(pericytesf)

print(
    pericytesf.obs[LOCATION_COL]
    .value_counts()
)

# =========================
# FILTER LOW GENES
# =========================
sc.pp.filter_genes(
    pericytesf,
    min_cells=5
)

# =========================
# DIFFERENTIAL EXPRESSION
# =========================
sc.tl.rank_genes_groups(
    pericytesf,
    groupby=LOCATION_COL,
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    use_raw=False
)

# =========================
# EXTRACT
# =========================
deg = sc.get.rank_genes_groups_df(
    pericytesf,
    group="Islet"
)

deg = deg.dropna(
    subset=["logfoldchanges", "pvals_adj"]
)

# =========================
# REMOVE CONTAMINATION
# =========================
contam_genes = [

    # endocrine
    "INS","IAPP","GCG","SST","PPY",
    "CHGA","CHGB",
    "PCSK1","PCSK2",
    "MAFA","MAFB","NKX6-1",
    "ISL1","UCN3","PAX6","SSTR2",
    "ADCYAP1","SYP","UCHL1",
    "NEUROD1","GJD2","GCK", "VAT1L", "SYP", "CD79A", "SYN1",
    # acinar
    "PRSS1","PRSS2","CPA1","CPA2",
    "CTRB1","CTRB2","CELA3A","CELA3B",
    "CPB1","REG1A","REG1B", "ALB", "CD24", "SDC4", "IGFBP7",

    # ductal
    "KRT19","KRT8","KRT18",
    "KRT17","KRT7","EPCAM","SOX9","CFTR"
]

deg = deg[
    ~deg["names"].isin(contam_genes)
].copy()

print(f"\nGenes remaining after cleanup: {deg.shape[0]}")

# =========================
# SAFE TRANSFORM
# =========================
deg["-log10(adj_p)"] = -np.log10(
    deg["pvals_adj"] + 1e-300
)

deg["-log10(adj_p)"] = (
    deg["-log10(adj_p)"]
    .clip(upper=20)
)

# =========================
# SIGNIFICANCE
# =========================
deg["significance"] = "NS"

deg.loc[
    (
        deg["logfoldchanges"] > fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Islet"

deg.loc[
    (
        deg["logfoldchanges"] < -fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Exocrine"

# =========================
# COLORS
# =========================
palette = {
    "Up in Islet": "#E1A7F8",
    "Up in Exocrine": "#6A0DAD",
    "NS": "lightgray"
}

# =========================
# PLOT
# =========================
fig, ax = plt.subplots(
    figsize=(4,3)
)

plot_order = [
    "NS",
    "Up in Exocrine",
    "Up in Islet"
]

for key in plot_order:

    group = deg[
        deg["significance"] == key
    ]

    ax.scatter(
        group["logfoldchanges"],
        group["-log10(adj_p)"],
        s=8,
        c=palette[key],
        label=key,
        alpha=0.8,
        rasterized=True
    )

# =========================
# THRESHOLDS
# =========================
ax.axhline(
    -np.log10(p_thresh),
    linestyle="--",
    color="black",
    linewidth=0.8
)

ax.axvline(
    fc_thresh,
    linestyle="--",
    color="black",
    linewidth=0.8
)

ax.axvline(
    -fc_thresh,
    linestyle="--",
    color="black",
    linewidth=0.8
)

# =========================
# LABEL TOP GENES
# =========================
# =========================
# LABEL ALL SIGNIFICANT GENES
# =========================
label_df = deg[
    deg["significance"] != "NS"
].copy()

print(f"\nSignificant genes labeled: {label_df.shape[0]}")

for _, row in label_df.iterrows():

    color = (
        "#E1A7F8"
        if row["significance"] == "Up in Islet"
        else "#6A0DAD"
    )

    ax.text(
        row["logfoldchanges"],
        row["-log10(adj_p)"],
        row["names"],
        fontsize=5,
        ha="center",
        va="bottom",
        color=color,
        fontweight="bold"
    )
# =========================
# FORMATTING
# =========================
ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel("-log10(adj. p-value)")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=6
)

plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "volcano_islet_vs_exocrine_periocytedscleaned"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================
# SAVE DEG
# =========================
deg.to_csv(
    f"{out_base}_DEG.csv",
    index=False
)

print("✔ DONE")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
LOCATION_COL = "Location"

fc_thresh = 0.2
p_thresh = 0.05

# =========================
# CHECK INPUT
# =========================

print(pericytesf)

print(
    pericytesf.obs[LOCATION_COL]
    .value_counts()
)

# =========================
# FILTER LOW GENES
# =========================

sc.pp.filter_genes(
    pericytesf,
    min_cells=5
)

# =========================
# DIFFERENTIAL EXPRESSION
# =========================

sc.tl.rank_genes_groups(
    pericytesf,
    groupby=LOCATION_COL,
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    use_raw=False
)

# =========================
# EXTRACT RESULTS
# =========================

deg = sc.get.rank_genes_groups_df(
    pericytesf,
    group="Islet"
)

deg = deg.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

# =========================
# REMOVE CONTAMINATION
# =========================




print(
    f"\nGenes remaining after cleanup: "
    f"{deg.shape[0]}"
)

# =========================
# SAFE TRANSFORM
# =========================

deg["-log10(adj_p)"] = -np.log10(
    deg["pvals_adj"] + 1e-300
)

deg["-log10(adj_p)"] = (
    deg["-log10(adj_p)"]
    .clip(upper=20)
)

# =========================
# SIGNIFICANCE
# =========================

deg["significance"] = "NS"

deg.loc[
    (
        deg["logfoldchanges"] > fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Islet"

deg.loc[
    (
        deg["logfoldchanges"] < -fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Exocrine"

# =========================
# COLORS
# =========================

palette = {
    "Up in Islet": "#E1A7F8",
    "Up in Exocrine": "#6A0DAD",
    "NS": "lightgray"
}

# =========================
# PLOT
# =========================

fig, ax = plt.subplots(
    figsize=(4,3)
)

plot_order = [
    "NS",
    "Up in Exocrine",
    "Up in Islet"
]

for key in plot_order:

    group = deg[
        deg["significance"] == key
    ]

    ax.scatter(
        group["logfoldchanges"],
        group["-log10(adj_p)"],
        s=8,
        c=palette[key],
        label=key,
        alpha=0.8,
        rasterized=True
    )

# =========================
# THRESHOLD LINES
# =========================

ax.axhline(
    -np.log10(p_thresh),
    linestyle="--",
    color="black",
    linewidth=0.8
)

ax.axvline(
    fc_thresh,
    linestyle="--",
    color="black",
    linewidth=0.8
)

ax.axvline(
    -fc_thresh,
    linestyle="--",
    color="black",
    linewidth=0.8
)

# =========================
# GENES TO LABEL
# =========================

genes_to_label = [

    # islet-enriched BM
    "C1S",
    "C1R",
    "COL6A3",
    "COL12A1",
    "IGFBP2",
    "PRELP",
    "LAMB1",
    "PCOLCE",
    "COL2A1",
    "SERPINA1",
    "LAMA4",

    # exocrine-enriched ECM
    "RGS5",
    "LAMC2",
    "MSLN2",
    "GLP1R",
    "ACE2",
    "FGA",

]

# =========================
# LABEL SUBSET
# =========================

label_df = deg[
    deg["names"].isin(
        genes_to_label
    )
].copy()

print(
    f"\nGenes labeled: "
    f"{label_df.shape[0]}"
)

print(
    label_df["names"].tolist()
)

# =========================
# LABEL GENES
# =========================

for _, row in label_df.iterrows():

    color = (
        "#E1A7F8"
        if row["significance"] == "Up in Islet"
        else "#6A0DAD"
    )

    ax.text(
        row["logfoldchanges"],
        row["-log10(adj_p)"],
        row["names"],
        fontsize=6,
        ha="center",
        va="bottom",
        color=color,
        fontweight="bold"
    )

# =========================
# FORMATTING
# =========================

ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel(
    "-log10(adj. p-value)"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=6
)

plt.tight_layout()

# =========================
# EXPORT
# =========================

out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "volcano_islet_vs_exocrine_pericytes_cleaned"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================
# SAVE DEG TABLE
# =========================

deg.to_csv(
    f"{out_base}_DEG.csv",
    index=False
)

print("\n✔ DONE")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================

LOCATION_COL = "Location"

fc_thresh = 0.2
p_thresh = 0.05

# =========================
# CHECK INPUT
# =========================

print(pericytesf)

print(
    pericytesf.obs[LOCATION_COL]
    .value_counts()
)

# =========================
# FILTER LOW GENES
# =========================

sc.pp.filter_genes(
    pericytesf,
    min_cells=5
)

# =========================
# DIFFERENTIAL EXPRESSION
# =========================

sc.tl.rank_genes_groups(
    pericytesf,
    groupby=LOCATION_COL,
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    use_raw=False
)

# =========================
# EXTRACT RESULTS
# =========================

deg = sc.get.rank_genes_groups_df(
    pericytesf,
    group="Islet"
)

deg = deg.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

# =========================
# SAFE TRANSFORM
# =========================

deg["-log10(adj_p)"] = -np.log10(
    deg["pvals_adj"] + 1e-300
)

deg["-log10(adj_p)"] = (
    deg["-log10(adj_p)"]
    .clip(upper=20)
)

# =========================
# SIGNIFICANCE
# =========================

deg["significance"] = "NS"

deg.loc[
    (
        deg["logfoldchanges"] > fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Islet"

deg.loc[
    (
        deg["logfoldchanges"] < -fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Exocrine"

# =========================
# GENES TO LABEL / SHOW
# =========================
genes_to_label = [

    # islet-enriched BM
    "C1S",
    "C1R",
    "COL6A3",
    "COL12A1",
    "IGFBP2",
    "PRELP",
    "LAMB1",
    "PCOLCE",
    "SERPINA1",

    # exocrine-enriched ECM
    "RGS5",
    "LAMC2",
    "MSLN2",
    "FLT1",
    "ACE2",
    "FGA",

]

# =========================
# KEEP ONLY SELECTED GENES
# =========================

plot_df = deg[
    deg["names"].isin(
        genes_to_label
    )
].copy()

print(
    f"\nGenes shown: "
    f"{plot_df.shape[0]}"
)

print(
    plot_df[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================
# COLORS
# =========================

palette = {
    "Up in Islet": "#E1A7F8",
    "Up in Exocrine": "#6A0DAD",
    "NS": "lightgray"
}

# =========================
# PLOT
# =========================

fig, ax = plt.subplots(
    figsize=(4,3)
)

# =========================
# PLOT ONLY SELECTED GENES
# =========================

for key in [
    "Up in Exocrine",
    "Up in Islet",
    "NS"
]:

    group = plot_df[
        plot_df["significance"] == key
    ]

    ax.scatter(
        group["logfoldchanges"],
        group["-log10(adj_p)"],
        s=30,
        c=palette[key],
        edgecolor="black",
        linewidth=0.3,
        label=key,
        alpha=0.95
    )

# =========================
# THRESHOLD LINES
# =========================



# =========================
# LABEL GENES
# =========================

for _, row in plot_df.iterrows():

    color = (
        "#E1A7F8"
        if row["significance"] == "Up in Islet"
        else "#6A0DAD"
    )

    ax.text(
        row["logfoldchanges"],
        row["-log10(adj_p)"] + 0.3,
        row["names"],
        fontsize=6,
        ha="center",
        va="bottom",
        color=color,
        fontweight="bold"
    )

# =========================
# FORMATTING
# =========================

ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel(
    "-log10(adj. p-value)"
)

ax.set_title(
    "Selected ECM-associated DEGs"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=6
)

plt.tight_layout()

# =========================
# EXPORT
# =========================

out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "Selected_Islet_vs_Exocrine_Pericyte_DEGs"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================
# SAVE TABLE
# =========================

plot_df.to_csv(
    f"{out_base}_table.csv",
    index=False
)

print("\n✔ DONE")

In [ ]:
# =========================
# SORT BY LOG FOLD CHANGE
# =========================

plot_df = plot_df.sort_values(
    "logfoldchanges"
)

# =========================
# PLOT AS ORDERED STRIP
# =========================

fig, ax = plt.subplots(
    figsize=(5,3)
)

# x positions
x = np.arange(plot_df.shape[0])

# colors
colors = [
    "#E1A7F8"
    if s == "Up in Islet"
    else "#6A0DAD"
    for s in plot_df["significance"]
]

# =========================
# SCATTER
# =========================

ax.scatter(
    x,
    plot_df["logfoldchanges"],
    s=45,
    c=colors,
    edgecolor="black",
    linewidth=0.4
)

# =========================
# LABELS
# =========================

for i, (_, row) in enumerate(
    plot_df.iterrows()
):

    ax.text(
        i,
        row["logfoldchanges"] + 0.1,
        row["names"],
        rotation=90,
        ha="center",
        va="bottom",
        fontsize=6,
        fontweight="bold",
        color=colors[i]
    )

# =========================
# CENTER LINE
# =========================

ax.axhline(
    0,
    linestyle="--",
    color="gray",
    linewidth=0.8
)

# =========================
# FORMATTING
# =========================

ax.set_xticks([])

ax.set_ylabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_title(
    "Selected pericyte niche-associated genes"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================
# EXPORT
# =========================

out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "Ordered_Selected_Pericyte_DEGs"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np

# =====================================================
# STYLE
# =====================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# SORT
# =====================================================

plot_df = plot_df.sort_values(
    "logfoldchanges"
).copy()

# =====================================================
# COLORS
# =====================================================

plot_df["color"] = np.where(
    plot_df["logfoldchanges"] > 0,
    "#E1A7F8",   # islet
    "#6A0DAD"    # exocrine
)

# =====================================================
# FIGURE
# =====================================================

fig, ax = plt.subplots(
    figsize=(4.5,4)
)

# y positions
y = np.arange(plot_df.shape[0])

# =====================================================
# LOLLIPOPS
# =====================================================

for i, row in enumerate(
    plot_df.itertuples()
):

    ax.hlines(
        y=i,
        xmin=0,
        xmax=row.logfoldchanges,
        color=row.color,
        linewidth=2
    )

    ax.scatter(
        row.logfoldchanges,
        i,
        s=45,
        color=row.color,
        edgecolor="black",
        linewidth=0.4,
        zorder=3
    )

# =====================================================
# LABELS
# =====================================================

ax.set_yticks(y)

ax.set_yticklabels(
    plot_df["names"],
    fontweight="bold"
)

# color labels
for tick, color in zip(
    ax.get_yticklabels(),
    plot_df["color"]
):
    tick.set_color(color)

# =====================================================
# CENTER LINE
# =====================================================

ax.axvline(
    0,
    linestyle="--",
    color="gray",
    linewidth=0.8
)

# =====================================================
# FORMAT
# =====================================================

ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel("")

ax.set_title(
    "Pericyte niche-associated genes"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =====================================================
# EXPORT
# =====================================================

out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "Pericyte_Niche_Lollipop"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# =====================================================
# STYLE
# =====================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# FIGURE
# =====================================================

fig, ax = plt.subplots(
    figsize=(4,3)
)

# =====================================================
# COLORS
# =====================================================

plot_df["color"] = np.where(
    plot_df["logfoldchanges"] > 0,
    "#D79BF5",
    "#5B1A9E"
)

# =====================================================
# SCATTER
# =====================================================

ax.scatter(
    plot_df["logfoldchanges"],
    plot_df["-log10(adj_p)"],
    s=45,
    c=plot_df["color"],
    edgecolor="black",
    linewidth=0.4,
    zorder=3
)

# =====================================================
# LABELS
# =====================================================

for _, row in plot_df.iterrows():

    ax.text(
        row["logfoldchanges"],
        row["-log10(adj_p)"] + 0.15,
        row["names"],
        fontsize=6,
        ha="center",
        va="bottom",
        fontweight="bold",
        color=row["color"]
    )

# =====================================================
# DIAGONAL GUIDES
# =====================================================

x = np.linspace(-2, 2.5, 100)

# right diagonal
ax.plot(
    x,
    0.8 * np.abs(x) + 0.3,
    linestyle="--",
    color="gray",
    linewidth=0.8,
    alpha=0.7
)

# =====================================================
# CENTER LINE
# =====================================================

ax.axvline(
    0,
    linestyle="--",
    color="lightgray",
    linewidth=0.8
)

# =====================================================
# FORMAT
# =====================================================

ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel(
    "-log10(adj. p-value)"
)

ax.set_title(
    "Selected pericyte niche-associated genes"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =====================================================
# EXPORT
# =====================================================

out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "Diagonal_Selected_Pericyte_DEGs"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# =====================================================
# STYLE
# =====================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# SORT
# =====================================================

plot_simple = plot_df.sort_values(
    "logfoldchanges"
).copy()

# =====================================================
# COLORS
# =====================================================

plot_simple["color"] = np.where(
    plot_simple["logfoldchanges"] > 0,
    "#D79BF5",   # islet
    "#5B1A9E"    # exocrine
)

# =====================================================
# FAKE Y POSITIONS
# slight jitter for aesthetics
# =====================================================

np.random.seed(0)

plot_simple["y"] = np.random.normal(
    loc=0,
    scale=0.08,
    size=plot_simple.shape[0]
)

# =====================================================
# FIGURE
# =====================================================

fig, ax = plt.subplots(
    figsize=(4,3)
)

# =====================================================
# SCATTER
# =====================================================

ax.scatter(
    plot_simple["logfoldchanges"],
    plot_simple["y"],
    s=55,
    c=plot_simple["color"],
    edgecolor="black",
    linewidth=0.4,
    zorder=3
)

# =====================================================
# LABELS
# =====================================================

for _, row in plot_simple.iterrows():

    ax.text(
        row["logfoldchanges"],
        row["y"] + 0.03,
        row["names"],
        fontsize=6,
        ha="center",
        va="bottom",
        fontweight="bold",
        color=row["color"]
    )

# =====================================================
# CENTER LINE
# =====================================================

ax.axvline(
    0,
    linestyle="--",
    color="gray",
    linewidth=0.8
)

# =====================================================
# FORMAT
# =====================================================

ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_yticks([])

ax.set_ylabel("")

ax.set_title(
    "Pericyte niche-associated genes"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout()

# =====================================================
# EXPORT
# =====================================================

out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "Simple_LogFC_Pericyte_Genes"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np

# =====================================================
# STYLE
# =====================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# COPY + SORT
# =====================================================

plot_bar = plot_df.copy()

plot_bar = plot_bar.sort_values(
    "logfoldchanges"
)

# =====================================================
# COLORS
# =====================================================

plot_bar["color"] = np.where(
    plot_bar["logfoldchanges"] > 0,
    "#EF0B0B",   # islet
    "#A001C0"    # exocrine
)



# =====================================================
# FIGURE
# =====================================================

fig, ax = plt.subplots(
    figsize=(2.5,2.5)
)

# =====================================================
# BARPLOT
# =====================================================

ax.barh(
    y=plot_bar["names"],
    width=plot_bar["logfoldchanges"],
    color=plot_bar["color"],
    edgecolor="black",
    linewidth=0.4
)

# =====================================================
# CENTER LINE
# =====================================================

ax.axvline(
    0,
    linestyle="--",
    color="gray",
    linewidth=0.8
)

# =====================================================
# COLOR Y LABELS
# =====================================================

for tick, color in zip(
    ax.get_yticklabels(),
    plot_bar["color"]
):

    tick.set_color(color)
    tick.set_fontweight("bold")

# =====================================================
# FORMAT
# =====================================================

ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel("")

ax.set_title(
    "Pericyte niche-associated genes"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =====================================================
# EXPORT
# =====================================================

out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "Pericyte_Niche_Barplot"
)

fig.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
#Pericytes from the Craig_Schapiro Dataset testing of the same genes above from spatial panel 

# =====================================================
# SUBSET TO PERICYTES
# =====================================================

peri = adata_new_ref[
    adata_new_ref.obs["cell_type_merged"] == "Pericyte"
].copy()

# =====================================================
# DEG: ISLET vs EXOCRINE
# =====================================================

sc.tl.rank_genes_groups(
    peri,
    groupby="Location",
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    use_raw=False
)

deg = sc.get.rank_genes_groups_df(
    peri,
    group="Islet"
)

# =====================================================
# GENES FROM SPATIAL ANALYSIS
# =====================================================

genes_plot = [
    "RGS5",
    "ACE2",
    "FLT1",
    "SERPINA1",
    "FGA",
    "LAMC2",
    "PCOLCE",
    "COL6A3",
    "COL12A1",
    "PRELP",
    "LAMB1",
    "C1S",
    "C1R",
    "IGFBP2"
]

# =====================================================
# SUBSET TO GENES
# =====================================================

plot_df = deg[
    deg["names"].isin(genes_plot)
].copy()

# keep same order as original figure
plot_df["names"] = pd.Categorical(
    plot_df["names"],
    categories=genes_plot,
    ordered=True
)

plot_df = plot_df.sort_values("logfoldchanges")

# =====================================================
# COLORS
# =====================================================

plot_df["color"] = np.where(
    plot_df["logfoldchanges"] > 0,
    "#EF0B0B",   # Islet enriched
    "#A001C0"    # Exocrine enriched
)

# =====================================================
# STYLE
# =====================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# PLOT
# =====================================================

fig, ax = plt.subplots(figsize=(2.5,2.5))

ax.barh(
    y=plot_df["names"],
    width=plot_df["logfoldchanges"],
    color=plot_df["color"],
    edgecolor="black",
    linewidth=0.4
)

ax.axvline(
    0,
    linestyle="--",
    color="gray",
    linewidth=0.8
)

# color gene labels
for tick, color in zip(
    ax.get_yticklabels(),
    plot_df["color"]
):
    tick.set_color(color)
    tick.set_fontweight("bold")

ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel("")
ax.set_title("Craig-Shapiro Pericytes")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    "/Users/kmeneses/Desktop/CraigShapiro_Pericyte_Niche_Genes.svg"
)

plt.savefig(
    "/Users/kmeneses/Desktop/CraigShapiro_Pericyte_Niche_Genes.pdf"
)

plt.savefig(
    "/Users/kmeneses/Desktop/CraigShapiro_Pericyte_Niche_Genes.png",
    dpi=600
)

plt.show()

print(
    plot_df[
        ["names","logfoldchanges","pvals_adj"]
    ]
)

In [ ]:
# =========================
# SUBSET TO PERICYTES
# =========================

pericytes_cshp = adata_new_ref[
    adata_new_ref.obs["cell_type"].isin(["Pericyte"])
].copy()

print(pericytes_cshp)

print(
    pericytes_cshp.obs["Location"]
    .value_counts()
)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================

LOCATION_COL = "Location"

fc_thresh = 0.2
p_thresh = 0.05

# =========================
# CHECK INPUT
# =========================

print(pericytes_cshp)

print(
    pericytes_cshp.obs[LOCATION_COL]
    .value_counts()
)

# =========================
# FILTER LOW GENES
# =========================

sc.pp.filter_genes(
    pericytes_cshp,
    min_cells=5
)

# =========================
# DIFFERENTIAL EXPRESSION
# =========================

sc.tl.rank_genes_groups(
    pericytes_cshp,
    groupby=LOCATION_COL,
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    use_raw=False
)

# =========================
# EXTRACT RESULTS
# =========================

deg = sc.get.rank_genes_groups_df(
    pericytes_cshp,
    group="Islet"
)

deg = deg.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

# =========================
# SAFE TRANSFORM
# =========================

deg["-log10(adj_p)"] = -np.log10(
    deg["pvals_adj"] + 1e-300
)

deg["-log10(adj_p)"] = (
    deg["-log10(adj_p)"]
    .clip(upper=20)
)

# =========================
# SIGNIFICANCE
# =========================

deg["significance"] = "NS"

deg.loc[
    (
        deg["logfoldchanges"] > fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Islet"

deg.loc[
    (
        deg["logfoldchanges"] < -fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Exocrine"

# =========================
# GENES TO LABEL / SHOW
# =========================
genes_to_label = [

    # islet-enriched BM
    "C1S",
    "C1R",
    "COL6A3",
    "COL12A1",
    "IGFBP2",
    "PRELP",
    "LAMB1",
    "PCOLCE",
    "SERPINA1",

    # exocrine-enriched ECM
    "RGS5",
    "LAMC2",
    "MSLN2",
    "FLT1",
    "ACE2",
    "FGA",

]

# =========================
# KEEP ONLY SELECTED GENES
# =========================

plot_df = deg[
    deg["names"].isin(
        genes_to_label
    )
].copy()

print(
    f"\nGenes shown: "
    f"{plot_df.shape[0]}"
)

print(
    plot_df[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================
# COLORS
# =========================

palette = {
    "Up in Islet": "#E1A7F8",
    "Up in Exocrine": "#6A0DAD",
    "NS": "lightgray"
}

# =========================
# PLOT
# =========================

fig, ax = plt.subplots(
    figsize=(4,3)
)

# =========================
# PLOT ONLY SELECTED GENES
# =========================

for key in [
    "Up in Exocrine",
    "Up in Islet",
    "NS"
]:

    group = plot_df[
        plot_df["significance"] == key
    ]

    ax.scatter(
        group["logfoldchanges"],
        group["-log10(adj_p)"],
        s=30,
        c=palette[key],
        edgecolor="black",
        linewidth=0.3,
        label=key,
        alpha=0.95
    )

# =========================
# THRESHOLD LINES
# =========================



# =========================
# LABEL GENES
# =========================

for _, row in plot_df.iterrows():

    color = (
        "#E1A7F8"
        if row["significance"] == "Up in Islet"
        else "#6A0DAD"
    )

    ax.text(
        row["logfoldchanges"],
        row["-log10(adj_p)"] + 0.3,
        row["names"],
        fontsize=6,
        ha="center",
        va="bottom",
        color=color,
        fontweight="bold"
    )

# =========================
# FORMATTING
# =========================

ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel(
    "-log10(adj. p-value)"
)

ax.set_title(
    "Selected ECM-associated DEGs"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=6
)

plt.tight_layout()

# =========================
# EXPORT
# =========================


plt.show()

# =========================
# SAVE TABLE
# =========================

plot_df.to_csv(
    f"{out_base}_table.csv",
    index=False
)

print("\n✔ DONE")

In [ ]:
# =========================
# REMOVE THRESHOLD LINES
# =========================

# DELETE THESE BLOCKS:

# ax.axhline(
#     -np.log10(p_thresh),
#     linestyle="--",
#     color="black",
#     linewidth=0.8
# )

# ax.axvline(
#     fc_thresh,
#     linestyle="--",
#     color="black",
#     linewidth=0.8
# )

# ax.axvline(
#     -fc_thresh,
#     linestyle="--",
#     color="black",
#     linewidth=0.8
# )

# =========================
# OPTIONAL:
# ADD CENTER LINE ONLY
# (cleaner)
# =========================

ax.axvline(
    0,
    linestyle="--",
    color="gray",
    linewidth=0.8,
    alpha=0.7
)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# STYLE
# =====================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# SUBSET:
# PERICYTES ONLY
# =====================================================

peri = vasc_healthy[
    vasc_healthy.obs["cell_type"] == "Pericytes"
].copy()

peri = peri[
    peri.obs["Location"].isin([
        "Islet",
        "Exocrine"
    ])
].copy()

# =====================================================
# CURATED GENE SETS
# =====================================================

bm_genes = [
    "COL4A1",
    "COL4A2",
    "COL4A3",
    "LAMA4",
    "LAMA5",
    "LAMB1",
    "LAMB2",
    "LAMC1",
    "LAMC3",
    "HSPG2",
    "AGRN",
    "LOXL4"
]

ecm_genes = [
    "COL1A1",
    "COL1A2",
    "COL3A1",
    "COL5A1",
    "COL5A2",
    "FN1",
    "VCAN",
    "DCN",
    "LUM",
    "POSTN"
]

plot_genes = [
    g for g in (bm_genes + ecm_genes)
    if g in peri.var_names
]

# =====================================================
# DOTPLOT
# =====================================================

sc.pl.dotplot(
    peri,
    var_names=plot_genes,
    groupby="Location",
    standard_scale="var",
    cmap="RdBu_r",
    dendrogram=False,
    swap_axes=False,
    figsize=(8,2.2),
    use_raw=False,
    show=False
)

# =====================================================
# FORMAT
# =====================================================

fig = plt.gcf()

for ax in fig.axes:

    ax.tick_params(
        axis="x",
        rotation=90,
        labelsize=7
    )

    ax.tick_params(
        axis="y",
        labelsize=8
    )

plt.tight_layout()

# =====================================================
# EXPORT
# =====================================================

out_base = (
    "/Users/kmeneses/Desktop/Fig_plots/"
    "Pericyte_Islet_Exocrine_Dotplot"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# STYLE
# =====================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# SUBSET
# =====================================================

peri = vasc_healthy[
    vasc_healthy.obs["cell_type"] == "Pericytes"
].copy()

peri = peri[
    peri.obs["Location"].isin([
        "Islet",
        "Exocrine"
    ])
].copy()

# =====================================================
# GENE SETS
# =====================================================

ecm_genes = [
    "COL4A3",
    "COL18A1",
    "LAMA4",
    "LAMC2",
    "COL6A3",
    "COL12A1"
    "COL2A1",
    "PRELP",
    "PCOLCE",
    "LAMB3", 
    "LAMB1"
]

other_genes = [
    "SERPINA1",
    "ACE2",
    "RGS5",
    "CALCA",
    "C1S",
    "C1R",
    "IGFBP2",
   
]

ecm_genes = [
    g for g in ecm_genes
    if g in peri.var_names
]

other_genes = [
    g for g in other_genes
    if g in peri.var_names
]

# =====================================================
# PANEL 1:
# BM GENES
# =====================================================

sc.pl.matrixplot(
    peri,
    var_names=ecm_genes,
    groupby="Location",
    cmap="magma",
    standard_scale=None,
    dendrogram=False,
    use_raw=False,
    figsize=(5,1.5),
    show=False
)

plt.title(
    "Basement membrane program"
)

plt.tight_layout()

plt.savefig(
    "/Users/kmeneses/Desktop/Fig_plots/BM_program_matrixplot.svg",
    bbox_inches="tight"
)

plt.show()

# =====================================================
# PANEL 2:
# INTERSTITIAL ECM
# =====================================================

sc.pl.matrixplot(
    peri,
    var_names=other_genes,
    groupby="Location",
    cmap="viridis",
    standard_scale=None,
    dendrogram=False,
    use_raw=False,
    figsize=(5,1.5),
    show=False
)

plt.title(
    "Interstitial ECM program"
)

plt.tight_layout()

plt.savefig(
    "/Users/kmeneses/Desktop/Fig_plots/Interstitial_ECM_matrixplot.svg",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# SETTINGS
# =========================
LOCATION_COL = "Location"

fc_thresh = 1
p_thresh = 0.05

# genes to label
genes_to_label = [
    "LAMA4",
    "ADAM9",
    "COL18A1",
    "LAMB2",
    "LAMB1",
    "C1R",
    "C1S",
    "COL12A1",
    "PCOLCE",
    "PREPL",
    "ITH5",
    "IGFBP2", 
    "LAMA2", 
    'ALB', 

]

# =========================
# SUBSET:
# HEALTHY PERICYTES

print(pericytesf)

print(
    pericytesf.obs[LOCATION_COL]
    .value_counts()
)

# =========================
# FILTER LOW GENES
# =========================
sc.pp.filter_genes(
    pericytesf,
    min_cells=5
)

# =========================
# LOG TRANSFORM
# =========================
if "log1p" not in pericytesf.uns:
    sc.pp.log1p(pericytesf)

# =========================
# DIFFERENTIAL EXPRESSION
# =========================
sc.tl.rank_genes_groups(
    pericytesf,
    groupby=LOCATION_COL,
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    use_raw=False
)

# =========================
# EXTRACT
# =========================
deg = sc.get.rank_genes_groups_df(
    pericytesf,
    group="Islet"
)

deg = deg.dropna(
    subset=["logfoldchanges", "pvals_adj"]
)

# =========================
# REMOVE CONTAMINATION
# =========================



# =========================
# SAFE TRANSFORM
# =========================
deg["-log10(FDR)"] = -np.log10(
    deg["pvals_adj"] + 1e-300
)

deg["-log10(FDR)"] = (
    deg["-log10(FDR)"]
    .clip(upper=20)
)

# =========================
# SIGNIFICANCE
# =========================
deg["significance"] = "NS"

deg.loc[
    (
        deg["logfoldchanges"] > fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Islet"

deg.loc[
    (
        deg["logfoldchanges"] < -fc_thresh
    ) &
    (
        deg["pvals_adj"] < p_thresh
    ),
    "significance"
] = "Up in Exocrine"

# =========================
# COLORS
# =========================
palette = {
    "Up in Islet": "#E1A7F8",
    "Up in Exocrine": "#6A0DAD",
    "NS": "lightgray"
}

# =========================
# PLOT
# =========================
fig, ax = plt.subplots(
    figsize=(3.5, 2.5)
)

plot_order = [
    "NS",
    "Up in Exocrine",
    "Up in Islet"
]

for key in plot_order:

    group = deg[
        deg["significance"] == key
    ]

    ax.scatter(
        group["logfoldchanges"],
        group["-log10(FDR)"],
        s=8,
        c=palette[key],
        label=key,
        alpha=0.8,
        rasterized=True
    )

# =========================
# THRESHOLDS
# =========================
ax.axhline(
    -np.log10(p_thresh),
    linestyle="--",
    color="black",
    linewidth=0.8
)

ax.axvline(
    fc_thresh,
    linestyle="--",
    color="black",
    linewidth=0.8
)

ax.axvline(
    -fc_thresh,
    linestyle="--",
    color="black",
    linewidth=0.8
)

# =========================
# LABEL SELECTED GENES
# =========================
label_df = deg[
    deg["names"].isin(genes_to_label)
].copy()

print("\nGenes labeled:")
print(label_df["names"].tolist())

for _, row in label_df.iterrows():

    color = (
        "#E1A7F8"
        if row["logfoldchanges"] > 0
        else "#6A0DAD"
    )

    ax.text(
        row["logfoldchanges"],
        row["-log10(FDR)"],
        row["names"],
        fontsize=6,
        ha="center",
        va="bottom",
        color=color,
        fontweight="bold"
    )

# =========================
# FORMATTING
# =========================
ax.set_xlabel(
    "log2 Fold Change\n(Islet vs Exocrine)"
)

ax.set_ylabel("-log10(FDR)")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    fontsize=6
)

plt.tight_layout()

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Fig_plots/volcano_islet_vs_exocrine_pericytes"

fig.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# =========================
# SAVE DEG
# =========================
deg.to_csv(
    f"{out_base}_DEG.csv",
    index=False
)

print("✔ DONE")

In [ ]:
bm_genes = [
    # Type IV collagens
    "COL4A1","COL4A2", "LAMB2","LAMB1",

    # Laminins
  "LAMA5",
   
    "LAMC1","LAMC3", "LAMA4", "COL4A3"]

In [ ]:
ECM = ['SFN', 'COL4A3', 'COL18A1', 'COL15A1', 'COL5A3', 'COL6A3', 'COL6A2', 'COL5A2', 'COL1A1', 'COL1A2', 'COL6A1', 'COL3A1', 
       'LGALS4', 'LOXL4', 'SFRP2', 'SPARCL1', 'IGFBP7', 'AGPS', 'AGRN', 'CLASRP', 'MGP', 'AMBP', 'HSPG2', 
       'F13A1', 'ASPN', 'LAMC3', 'TINAGL1', 'VWA1', 'PCOLCE', 'MFGE8', 'FBLN2', 
       'LAMB2', 'FBN1', 'LAMA2', 'THBS1', 'VTN', 'FN1', 'FGG', 'FGB', 'FGA', 'LAMA5', 'COL16A1', 'COL11A1', 
       'COL4A2', 'COL4A1', 'COL2A1', 'OGN', 'EMILIN1', 'LAMA4', 'POSTN', 'FBN2', 'COL12A1', 'COL14A1', 'COL5A1', 'LAMC1', 'LAMC2',  'LAMB3', 'FBLN1',
        'LAMB1', 'LAMB4', 'LAMA3', 'SDC1']

In [ ]:
ecm_genes_clean = [
    g for g in ECM if g not in bm_genes
]

In [ ]:
pericytesf

In [ ]:
pericytesf.obs['Location']

In [ ]:
ccd.ul.run_umap(pericytesf, source_key='Concord', result_key='Concord_UMAPm', n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine')

# Plot the UMAP embeddings
color_by = ['cell_type', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    pericytesf, basis='Concord_UMAPm', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data'
)

# Pericyte BM/ ECM Expression Analysis

Scoring ECM or BM gene set for all pericytes in samples and project scores in embedding 
Donor-level gene scoring comparison for significance testing 

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import matplotlib as mpl
import numpy as np
import os

# =========================
# STYLE (publication-ready)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# RESCORE ECM / BM
# =========================
valid_ecm = [g for g in ecm_genes_clean if g in pericytesf.var_names]
valid_bm  = [g for g in bm_genes if g in pericytesf.var_names]

sc.tl.score_genes(pericytesf, gene_list=valid_ecm, score_name="ECM_score", use_raw=False)
sc.tl.score_genes(pericytesf, gene_list=valid_bm,  score_name="BM_score",  use_raw=False)

# =========================
# MATCH COLOR SCALE
# =========================
vmin = min(
    np.percentile(pericytesf.obs["BM_score"], 5),
    np.percentile(pericytesf.obs["ECM_score"], 5)
)

vmax = max(
    np.percentile(pericytesf.obs["BM_score"], 95),
    np.percentile(pericytesf.obs["ECM_score"], 95)
)

# =========================
# LOCATION COLORS
# =========================
location_palette = {
    "Islet": "#A001C0",      # dark purple
    "Exocrine": "#EF0B0B"    # light purple
}

# =========================
# FIGURE
# =========================
genesets = ["Location", "BM_score", "ECM_score"]

fig, axes = plt.subplots(1, 3, figsize=(6, 2))

for ax, g in zip(axes, genesets):

    if g == "Location":
        sc.pl.embedding(
            pericytesf,
            basis="Concord_UMAPm",
            color=g,
            size=4,
            frameon=True,
            legend_loc=None,
            ax=ax,
            show=False,
            sort_order=True,
            palette=location_palette
        )
    else:
        sc.pl.embedding(
            pericytesf,
            basis="Concord_UMAPm",
            color=g,
            size=4,
            frameon=True,
            legend_loc=None,
            ax=ax,
            show=False,
            sort_order=True,
            vmin=vmin,
            vmax=vmax,
            cmap="viridis"
        )

    # rasterize points (smaller files)
    for coll in ax.collections:
        coll.set_rasterized(True)

    ax.set_title(g.replace("_", " "))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

plt.subplots_adjust(wspace=0.3)

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure_2_plots/SI/UMAP_BM_ECM_clean"

os.makedirs(os.path.dirname(out_base), exist_ok=True)

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight")

plt.show()

print("✔ DONE — UMAP BM/ECM figure saved")

In [ ]:
import scanpy as sc

# =========================
# Run DEG (same as before)
# =========================
sc.tl.rank_genes_groups(
    pericytesf,
    groupby="Location",
    groups=["Exocrine"],
    reference="Islet",
    method="wilcoxon",
    use_raw=False
)

# =========================
# Extract full DEG table
# =========================
exo_df = sc.get.rank_genes_groups_df(pericytesf, group="Exocrine")

# =========================
# 🔥 FILTER TO ECM GENES
# =========================
# assumes you already have:
# ECM_GENES (or use your ecm_filtered list)

valid_ecm = [g for g in ecm_genes_clean if g in pericytesf.var_names]

exo_ecm_df = exo_df[exo_df["names"].isin(valid_ecm)].copy()

# sort by strongest enrichment
exo_ecm_df = exo_ecm_df.sort_values("logfoldchanges", ascending=False)

# =========================
# View top ECM genes
# =========================
print(exo_ecm_df.head(20))

In [ ]:
import scanpy as sc

# =========================
# Run DEG (same as before)
# =========================
sc.tl.rank_genes_groups(
    pericytesf,
    groupby="Location",
    groups=["Islet"],
    reference="Exocrine",
    method="wilcoxon",
    use_raw=False
)

# =========================
# Extract full DEG table
# =========================
isl_df = sc.get.rank_genes_groups_df(pericytesf, group="Islet")

# =========================
# 🔥 FILTER TO ECM GENES
# =========================
# assumes you already have:
# ECM_GENES (or use your ecm_filtered list)

valid_ecm = [g for g in ecm_genes_clean if g in pericytesf.var_names]

isl_ecm_df = isl_df[isl_df["names"].isin(valid_ecm)].copy()

# sort by strongest enrichment
isl_ecm_df = isl_ecm_df.sort_values("logfoldchanges", ascending=False)

# =========================
# View top ECM genes
# =========================
print(isl_ecm_df.head(20))

In [ ]:
valid_ecm = [g for g in bm_genes if g in pericytesf.var_names]

exo_ecm_df = exo_df[exo_df["names"].isin(valid_ecm)].copy()

# sort by strongest enrichment
exo_ecm_df = exo_ecm_df.sort_values("logfoldchanges", ascending=False)

# =========================
# View top ECM genes
# =========================
print(exo_ecm_df.head(20))

In [ ]:
import os

# =========================
# FILTER ECM GENES
# =========================
valid_ecm = [g for g in bm_genes if g in pericytesf.var_names]

exo_ecm_df = exo_df[exo_df["names"].isin(valid_ecm)].copy()

# =========================
# SORT BY EFFECT SIZE
# =========================
exo_ecm_df = exo_ecm_df.sort_values("logfoldchanges", ascending=False)

# =========================
# VIEW TOP
# =========================
print(exo_ecm_df.head(20))

# =========================
# SAVE TO CSV
# =========================
out_path = "/Users/kmeneses/Desktop/Fig_plots/exo_ECM_genes.csv"

os.makedirs(os.path.dirname(out_path), exist_ok=True)

exo_ecm_df.to_csv(out_path, index=False)

print(f"✔ Saved to: {out_path}")

In [ ]:
bm_mean  = np.mean(pericytesf.obs["BM_score"])
ecm_mean = np.mean(pericytesf.obs["ECM_score"])

In [ ]:
bm_mean

In [ ]:
ecm_mean

In [ ]:
bm_thresh  = np.percentile(pericytesf.obs["BM_score"], 80)
ecm_thresh = np.percentile(pericytesf.obs["ECM_score"], 80)

In [ ]:
pericytesf.obs["BM_high"]  = pericytesf.obs["BM_score"]  > bm_thresh
pericytesf.obs["ECM_high"] = pericytesf.obs["ECM_score"] > ecm_thresh

In [ ]:
summary_donor = (
    pericytesf.obs
    .groupby(["Sample","Location"])[["BM_high","ECM_high"]]
    .mean()
    .reset_index()
)
summary_donor

In [ ]:
summary_donor = (
    pericytesf.obs
    .groupby(["Sample", "Location"])[["BM_high", "ECM_high"]]
    .mean()
    .reset_index()
)

# convert to percent
summary_donor[["BM_high","ECM_high"]] *= 100

summary_donor.head()

In [ ]:
from scipy.stats import wilcoxon

# Pivot for paired comparison
bm_pivot = summary_donor.pivot(index="Sample", columns="Location", values="BM_high")
ecm_pivot = summary_donor.pivot(index="Sample", columns="Location", values="ECM_high")

bm_stat = wilcoxon(bm_pivot["Islet"], bm_pivot["Exocrine"])
ecm_stat = wilcoxon(ecm_pivot["Islet"], ecm_pivot["Exocrine"])

print("BM p-value:", bm_stat.pvalue)
print("ECM p-value:", ecm_stat.pvalue)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "Arial",
    "svg.fonttype": "none",
    "pdf.fonttype": 42
})

fig, axes = plt.subplots(1, 2, figsize=(4.5,2.5))

for ax, program in zip(axes, ["BM_high","ECM_high"]):

    sns.boxplot(
        data=summary_donor,
        x="Location",
        y=program,
        ax=ax,
        showcaps=True,
        boxprops=dict(facecolor="white"),
        showfliers=False
    )

    sns.stripplot(
        data=summary_donor,
        x="Location",
        y=program,
        ax=ax,
        color="black",
        size=4
    )

    ax.set_ylabel("% High_Scoring Cells")
    ax.set_title(program.replace("_high"," High"))
    ax.set_xlabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
summary_donor

# Donor Level & Cell Level BM Gene Set Scoring 



In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

from scipy.stats import mannwhitneyu, wilcoxon

# ============================================================
# STYLE
# ============================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# ============================================================
# SETTINGS
# ============================================================
group_col = "cell_type_final"
donor_col = "Sample"

groups_keep = [
    "Pericytes",
    "Islet-associated Fibroblasts"
]

bm_genes = [
    "COL4A1","COL4A2","COL4A3",
    "LAMA4","LAMA5",
    "LAMB1","LAMB2",
    "LAMC1","LAMC3"
]

bm_score_name = "BM_score"

out_base = "/Users/kmeneses/Desktop/Fig_plots/islet_Pericyte_vs_PeriiFib_BM"

dpi = 600

# ============================================================
# SUBSET
# ============================================================
ad = islet_adata_vasc.copy()

# clean labels
ad.obs[group_col] = (
    ad.obs[group_col]
    .astype(str)
    .str.strip()
)

# subset desired groups
ad = ad[
    ad.obs[group_col]
    .isin(groups_keep)
].copy()

# categorical ordering
ad.obs[group_col] = pd.Categorical(
    ad.obs[group_col],
    categories=groups_keep,
    ordered=True
)

print("\nGroups present:")
print(ad.obs[group_col].value_counts())

print(f"\nCells kept: {ad.n_obs}")

# ============================================================
# BM GENES
# ============================================================
present = [
    g for g in bm_genes
    if g in ad.var_names
]

print(f"\nBM genes present ({len(present)}):")
print(present)

if len(present) < 4:
    raise ValueError("Too few BM genes detected.")

# ============================================================
# LOG TRANSFORM
# ============================================================
if "log1p" not in ad.uns:
    sc.pp.log1p(ad)

# ============================================================
# BM SCORE
# ============================================================
sc.tl.score_genes(
    ad,
    gene_list=present,
    score_name=bm_score_name,
    use_raw=False
)

# ============================================================
# CELL-LEVEL STATS
# ============================================================
x1 = ad.obs.loc[
    ad.obs[group_col] == groups_keep[0],
    bm_score_name
].values

x2 = ad.obs.loc[
    ad.obs[group_col] == groups_keep[1],
    bm_score_name
].values

p_cell = mannwhitneyu(
    x1,
    x2,
    alternative="two-sided"
).pvalue

# ============================================================
# DONOR-LEVEL PSEUDOBULK
# ============================================================
pb = (
    ad.obs[
        [donor_col, group_col, bm_score_name]
    ]
    .groupby(
        [donor_col, group_col],
        observed=True
    )[bm_score_name]
    .median()
    .unstack()
    .dropna()
)

p_paired = np.nan

if pb.shape[0] >= 3:

    p_paired = wilcoxon(
        pb[groups_keep[0]],
        pb[groups_keep[1]]
    ).pvalue

print(f"\nMWU p-value = {p_cell:.3e}")
print(f"Paired donor p-value = {p_paired:.3e}")

# ============================================================
# GENE-LEVEL SUMMARY
# ============================================================
X = ad[:, present].X

if hasattr(X, "toarray"):
    X = X.toarray()

mask_peri = (
    ad.obs[group_col] == groups_keep[0]
).values

mask_fib = (
    ad.obs[group_col] == groups_keep[1]
).values

rows = []

for i, g in enumerate(present):

    a = X[mask_peri, i]
    b = X[mask_fib, i]

    rows.append({
        "gene": g,
        "mean_pericytes": np.mean(a),
        "mean_peri_fib": np.mean(b),
        "diff": np.mean(a) - np.mean(b),
        "frac_on_pericytes": (a > 0).mean(),
        "frac_on_peri_fib": (b > 0).mean(),
    })

gene_summary = pd.DataFrame(rows)

gene_summary = gene_summary.sort_values(
    "diff",
    ascending=False
)

# ============================================================
# COLORS
# ============================================================
colors = {
    "Pericytes": "#FF00B8",
    "Islet-associated Fibroblasts": "#C68642"
}

# ============================================================
# FIGURE
# ============================================================
fig = plt.figure(figsize=(10.5, 3.6))

# ------------------------------------------------------------
# PANEL A — CELL LEVEL
# ------------------------------------------------------------
ax1 = plt.subplot(1,3,1)

ax1.boxplot(
    [x1, x2],
    labels=groups_keep,
    showfliers=False
)

ax1.scatter(
    np.random.normal(1, 0.05, len(x1)),
    x1,
    s=6,
    alpha=0.2,
    c=colors[groups_keep[0]]
)

ax1.scatter(
    np.random.normal(2, 0.05, len(x2)),
    x2,
    s=6,
    alpha=0.2,
    c=colors[groups_keep[1]]
)

ax1.set_title(
    f"BM score (cells)\nMWU p={p_cell:.1e}",
    fontweight="bold"
)

ax1.set_ylabel("BM score")

# ------------------------------------------------------------
# PANEL B — DONOR PAIRED
# ------------------------------------------------------------
ax2 = plt.subplot(1,3,2)

for _, row in pb.iterrows():

    ax2.plot(
        [1,2],
        [row[groups_keep[0]], row[groups_keep[1]]],
        lw=1,
        color="gray",
        alpha=0.7
    )

ax2.scatter(
    np.ones(len(pb)),
    pb[groups_keep[0]],
    s=50,
    c=colors[groups_keep[0]]
)

ax2.scatter(
    np.ones(len(pb))*2,
    pb[groups_keep[1]],
    s=50,
    c=colors[groups_keep[1]]
)

ax2.set_xticks([1,2])

ax2.set_xticklabels(
    groups_keep,
    rotation=15
)

ax2.set_title(
    f"Donor paired\np={p_paired:.1e}",
    fontweight="bold"
)

# ------------------------------------------------------------
# PANEL C — GENE SHIFTS
# ------------------------------------------------------------
ax3 = plt.subplot(1,3,3)

ax3.axvline(
    0,
    color="black",
    lw=1
)

ax3.scatter(
    gene_summary["diff"],
    np.arange(len(gene_summary)),
    s=25,
    color="black"
)

ax3.set_yticks(
    np.arange(len(gene_summary))
)

ax3.set_yticklabels(
    gene_summary["gene"]
)

ax3.set_title(
    "BM genes: mean shift",
    fontweight="bold"
)

ax3.set_xlabel(
    "Pericytes − Peri-islet Fibroblasts"
)

# ============================================================
# CLEANUP
# ============================================================
plt.tight_layout()

# ============================================================
# SAVE
# ============================================================
os.makedirs(
    os.path.dirname(out_base),
    exist_ok=True
)

fig.savefig(
    out_base + ".png",
    dpi=dpi,
    bbox_inches="tight"
)

fig.savefig(
    out_base + ".pdf",
    bbox_inches="tight"
)

fig.savefig(
    out_base + ".svg",
    bbox_inches="tight"
)

gene_summary.to_csv(
    out_base + "_genes.csv",
    index=False
)

pb.reset_index().to_csv(
    out_base + "_donors.csv",
    index=False
)

plt.show()

print("\n✔ DONE — clean BM analysis")

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

from scipy.stats import mannwhitneyu, wilcoxon

# ============================================================
# STYLE
# ============================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# ============================================================
# SETTINGS
# ============================================================
group_col = "cell_type_final"
donor_col = "Sample"

groups_keep = [
    "Pericytes",
    "Endothelial Cells"
]

bm_genes = [
    "COL4A1","COL4A2","COL4A3",
    "LAMA4","LAMA5",
    "LAMB1","LAMB2",
    "LAMC1","LAMC3"
]

bm_score_name = "BM_score"

out_base = "/Users/kmeneses/Desktop/Fig_plots/islet_Pericyte_vsECs_BM"

dpi = 600

# ============================================================
# SUBSET
# ============================================================
ad = islet_adata_vasc.copy()

# clean labels
ad.obs[group_col] = (
    ad.obs[group_col]
    .astype(str)
    .str.strip()
)

# subset desired groups
ad = ad[
    ad.obs[group_col]
    .isin(groups_keep)
].copy()

# categorical ordering
ad.obs[group_col] = pd.Categorical(
    ad.obs[group_col],
    categories=groups_keep,
    ordered=True
)

print("\nGroups present:")
print(ad.obs[group_col].value_counts())

print(f"\nCells kept: {ad.n_obs}")

# ============================================================
# BM GENES
# ============================================================
present = [
    g for g in bm_genes
    if g in ad.var_names
]

print(f"\nBM genes present ({len(present)}):")
print(present)

if len(present) < 4:
    raise ValueError("Too few BM genes detected.")

# ============================================================
# LOG TRANSFORM
# ============================================================
if "log1p" not in ad.uns:
    sc.pp.log1p(ad)

# ============================================================
# BM SCORE
# ============================================================
sc.tl.score_genes(
    ad,
    gene_list=present,
    score_name=bm_score_name,
    use_raw=False
)

# ============================================================
# CELL-LEVEL STATS
# ============================================================
x1 = ad.obs.loc[
    ad.obs[group_col] == groups_keep[0],
    bm_score_name
].values

x2 = ad.obs.loc[
    ad.obs[group_col] == groups_keep[1],
    bm_score_name
].values

p_cell = mannwhitneyu(
    x1,
    x2,
    alternative="two-sided"
).pvalue

# ============================================================
# DONOR-LEVEL PSEUDOBULK
# ============================================================
pb = (
    ad.obs[
        [donor_col, group_col, bm_score_name]
    ]
    .groupby(
        [donor_col, group_col],
        observed=True
    )[bm_score_name]
    .median()
    .unstack()
    .dropna()
)

p_paired = np.nan

if pb.shape[0] >= 3:

    p_paired = wilcoxon(
        pb[groups_keep[0]],
        pb[groups_keep[1]]
    ).pvalue

print(f"\nMWU p-value = {p_cell:.3e}")
print(f"Paired donor p-value = {p_paired:.3e}")

# ============================================================
# GENE-LEVEL SUMMARY
# ============================================================
X = ad[:, present].X

if hasattr(X, "toarray"):
    X = X.toarray()

mask_peri = (
    ad.obs[group_col] == groups_keep[0]
).values

mask_fib = (
    ad.obs[group_col] == groups_keep[1]
).values

rows = []

for i, g in enumerate(present):

    a = X[mask_peri, i]
    b = X[mask_fib, i]

    rows.append({
        "gene": g,
        "mean_pericytes": np.mean(a),
        "mean_peri_fib": np.mean(b),
        "diff": np.mean(a) - np.mean(b),
        "frac_on_pericytes": (a > 0).mean(),
        "frac_on_peri_fib": (b > 0).mean(),
    })

gene_summary = pd.DataFrame(rows)

gene_summary = gene_summary.sort_values(
    "diff",
    ascending=False
)

# ============================================================
# COLORS
# ============================================================
colors = {
    "Pericytes": "#FF00B8",
    "Endothelial Cells": "#33a02c"
}

# ============================================================
# FIGURE
# ============================================================
fig = plt.figure(figsize=(10.5, 3.6))

# ------------------------------------------------------------
# PANEL A — CELL LEVEL
# ------------------------------------------------------------
ax1 = plt.subplot(1,3,1)

ax1.boxplot(
    [x1, x2],
    labels=groups_keep,
    showfliers=False
)

ax1.scatter(
    np.random.normal(1, 0.05, len(x1)),
    x1,
    s=6,
    alpha=0.2,
    c=colors[groups_keep[0]]
)

ax1.scatter(
    np.random.normal(2, 0.05, len(x2)),
    x2,
    s=6,
    alpha=0.2,
    c=colors[groups_keep[1]]
)

ax1.set_title(
    f"BM score (cells)\nMWU p={p_cell:.1e}",
    fontweight="bold"
)

ax1.set_ylabel("BM score")

# ------------------------------------------------------------
# PANEL B — DONOR PAIRED
# ------------------------------------------------------------
ax2 = plt.subplot(1,3,2)

for _, row in pb.iterrows():

    ax2.plot(
        [1,2],
        [row[groups_keep[0]], row[groups_keep[1]]],
        lw=1,
        color="gray",
        alpha=0.7
    )

ax2.scatter(
    np.ones(len(pb)),
    pb[groups_keep[0]],
    s=50,
    c=colors[groups_keep[0]]
)

ax2.scatter(
    np.ones(len(pb))*2,
    pb[groups_keep[1]],
    s=50,
    c=colors[groups_keep[1]]
)

ax2.set_xticks([1,2])

ax2.set_xticklabels(
    groups_keep,
    rotation=15
)

ax2.set_title(
    f"Donor paired\np={p_paired:.1e}",
    fontweight="bold"
)

# ------------------------------------------------------------
# PANEL C — GENE SHIFTS
# ------------------------------------------------------------
ax3 = plt.subplot(1,3,3)

ax3.axvline(
    0,
    color="black",
    lw=1
)

ax3.scatter(
    gene_summary["diff"],
    np.arange(len(gene_summary)),
    s=25,
    color="black"
)

ax3.set_yticks(
    np.arange(len(gene_summary))
)

ax3.set_yticklabels(
    gene_summary["gene"]
)

ax3.set_title(
    "BM genes: mean shift",
    fontweight="bold"
)

ax3.set_xlabel(
    "Pericytes − ECs"
)

# ============================================================
# CLEANUP
# ============================================================
plt.tight_layout()

# ============================================================
# SAVE
# ============================================================
os.makedirs(
    os.path.dirname(out_base),
    exist_ok=True
)

fig.savefig(
    out_base + ".png",
    dpi=dpi,
    bbox_inches="tight"
)

fig.savefig(
    out_base + ".pdf",
    bbox_inches="tight"
)

fig.savefig(
    out_base + ".svg",
    bbox_inches="tight"
)

gene_summary.to_csv(
    out_base + "_genes.csv",
    index=False
)

pb.reset_index().to_csv(
    out_base + "_donors.csv",
    index=False
)

plt.show()

print("\n✔ DONE — clean BM analysis")